In [38]:
import sys
import pandas as pd

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Ambiente:", sys.executable)

Python: 3.12.10
Pandas: 2.2.3
Ambiente: c:\Users\andre\Documentos\Codes\Python\tech-challenge-fase3\tech-challenge-fase3\.venv\Scripts\python.exe


# Análise Exploratória de Dados (EDA) - Censo Escolar 2022 a 2024

## Objetivo
Analisar os microdados do Censo Escolar de 2022, 2023 e 2024 utilizando Pandas, avaliando estrutura, qualidade e distribuição das variáveis para fundamentar a seleção dos dados utilizados no Tech Challenge 3.

## Metodologia
Os arquivos serão analisados individualmente, preservando a separação por ano. Os critérios de filtragem e seleção serão documentados a partir dos resultados observados.

In [39]:
from pathlib import Path
import pandas as pd

# Localiza a raiz do projeto a partir do notebook
BASE_DIR = Path.cwd().resolve()
DADOS_DIR = BASE_DIR / "censo_escolar"

# Lista os arquivos CSV disponíveis
arquivos = sorted(DADOS_DIR.glob("*.csv"))

print(f"Diretório: {DADOS_DIR}")
print(f"Arquivos encontrados: {len(arquivos)}")

for arquivo in arquivos:
    tamanho_mb = arquivo.stat().st_size / (1024 ** 2)
    print(f"{arquivo.name}: {tamanho_mb:.2f} MB")

Diretório: C:\Users\andre\Documentos\Codes\Python\tech-challenge-fase3\tech-challenge-fase3\src\data\notebooks\censo_escolar
Arquivos encontrados: 7
2022_microdados_ed_basica.csv: 181.21 MB
2023_microdados_ed_basica.csv: 200.31 MB
2024_microdados_ed_basica.csv: 207.83 MB
2025_Tabela_Docente_V2.csv: 84.25 MB
2025_Tabela_Escola_V2.csv: 134.87 MB
2025_Tabela_Matricula_V2.csv: 117.61 MB
2025_Tabela_Turma_V2.csv: 95.29 MB


## 1. Inspeção inicial dos dados

Nesta etapa, verificamos a estrutura dos arquivos originais
do Censo Escolar de 2022, 2023 e 2024.

A inspeção inicial utiliza uma pequena amostra para validar
a leitura dos CSVs, sem carregar os arquivos completos
na memória.

Nenhum filtro ou transformação é aplicado nesta etapa.

In [40]:
# Inspeção inicial dos arquivos CSV

for arquivo in arquivos:
    ano = arquivo.name[:4]

    df_amostra = pd.read_csv(
        arquivo,
        sep=";",
        encoding="latin1",
        nrows=5
    )

    print(f"\n{'=' * 50}")
    print(f"CENSO ESCOLAR {ano}")
    print(f"{'=' * 50}")

    print(f"Colunas identificadas: {len(df_amostra.columns)}")
    print(f"Primeiras colunas: {df_amostra.columns[:10].tolist()}")
    print(f"Registros lidos: {len(df_amostra)}")


CENSO ESCOLAR 2022
Colunas identificadas: 385
Primeiras colunas: ['NU_ANO_CENSO', 'NO_REGIAO', 'CO_REGIAO', 'NO_UF', 'SG_UF', 'CO_UF', 'NO_MUNICIPIO', 'CO_MUNICIPIO', 'NO_MESORREGIAO', 'CO_MESORREGIAO']
Registros lidos: 5

CENSO ESCOLAR 2023
Colunas identificadas: 408
Primeiras colunas: ['NU_ANO_CENSO', 'NO_REGIAO', 'CO_REGIAO', 'NO_UF', 'SG_UF', 'CO_UF', 'NO_MUNICIPIO', 'CO_MUNICIPIO', 'NO_REGIAO_GEOG_INTERM', 'CO_REGIAO_GEOG_INTERM']
Registros lidos: 5

CENSO ESCOLAR 2024
Colunas identificadas: 426
Primeiras colunas: ['NU_ANO_CENSO', 'NO_REGIAO', 'CO_REGIAO', 'NO_UF', 'SG_UF', 'CO_UF', 'NO_MUNICIPIO', 'CO_MUNICIPIO', 'NO_REGIAO_GEOG_INTERM', 'CO_REGIAO_GEOG_INTERM']
Registros lidos: 5

CENSO ESCOLAR 2025
Colunas identificadas: 184
Primeiras colunas: ['NU_ANO_CENSO', 'NO_REGIAO', 'CO_REGIAO', 'NO_UF', 'SG_UF', 'CO_UF', 'NO_MUNICIPIO', 'CO_MUNICIPIO', 'NO_REGIAO_GEOG_INTERM', 'CO_REGIAO_GEOG_INTERM']
Registros lidos: 5

CENSO ESCOLAR 2025
Colunas identificadas: 290
Primeiras colunas: 

In [41]:
# Comparação das estruturas dos censos de 2022 a 2025

colunas_por_ano = {}

# Arquivos de 2022 a 2024
for ano in [2022, 2023, 2024]:
    arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

    df_amostra = pd.read_csv(
        arquivo,
        sep=";",
        encoding="latin1",
        nrows=0
    )

    colunas_por_ano[ano] = set(df_amostra.columns)

# Arquivo de escolas de 2025
arquivo_2025 = DADOS_DIR / "2025_Tabela_Escola_V2.csv"

df_amostra_2025 = pd.read_csv(
    arquivo_2025,
    sep=";",
    encoding="latin1",
    nrows=0
)

colunas_por_ano[2025] = set(df_amostra_2025.columns)

# Variáveis comuns aos quatro anos
colunas_comuns = set.intersection(
    *colunas_por_ano.values()
)

print(f"Variáveis comuns aos quatro anos: {len(colunas_comuns)}")

for ano in sorted(colunas_por_ano):
    exclusivas = colunas_por_ano[ano] - set.union(
        *(colunas for outro_ano, colunas in colunas_por_ano.items()
          if outro_ano != ano)
    )

    print(f"\n{ano}:")
    print(f"Total de variáveis: {len(colunas_por_ano[ano])}")
    print(f"Exclusivas deste ano: {len(exclusivas)}")
    print(f"Variáveis exclusivas: {sorted(exclusivas)}")

Variáveis comuns aos quatro anos: 222

2022:
Total de variáveis: 385
Exclusivas deste ano: 48
Variáveis exclusivas: ['IN_AGUA_FILTRADA', 'IN_BANHEIRO_DENTRO_PREDIO', 'IN_BANHEIRO_FORA_PREDIO', 'IN_BAS', 'IN_BERCARIO', 'IN_BRASIL_ALFABETIZADO', 'IN_CONVENIADA_PP', 'IN_DEPENDENCIAS_PNE', 'IN_ENERGIA_GERADOR', 'IN_ENERGIA_OUTROS', 'IN_EQUIP_FAX', 'IN_EQUIP_FOTO', 'IN_EQUIP_RETROPROJETOR', 'IN_EQUIP_VIDEOCASSETE', 'IN_FINAL_SEMANA', 'IN_FORMACAO_ALTERNANCIA', 'IN_FUNDAMENTAL_CICLOS', 'IN_GRUPOS_NAO_SERIADOS', 'IN_LAVANDERIA', 'IN_LIXO_JOGA_OUTRA_AREA', 'IN_LIXO_OUTROS', 'IN_LIXO_RECICLA', 'IN_LOCAL_FUNC_CASA_PROFESSOR', 'IN_LOCAL_FUNC_SALAS_EMPRESA', 'IN_LOCAL_FUNC_TEMPLO_IGREJA', 'IN_MATERIAL_ESP_INDIGENA', 'IN_MATERIAL_ESP_NAO_UTILIZA', 'IN_MATERIAL_ESP_QUILOMBOLA', 'IN_MODULOS', 'IN_PERIODOS_SEMESTRAIS', 'IN_SERIE_ANO', 'IN_TIPO_ATEND_AC', 'IN_TIPO_ATEND_AEE', 'IN_TIPO_ATEND_ESCOLARIZACAO', 'QT_COMPUTADOR', 'QT_COMP_ADMINISTRATIVO', 'QT_COMP_ALUNO', 'QT_EQUIP_COPIADORA', 'QT_EQUIP_FAX',

## 2. Seleção inicial das variáveis

Com base no objetivo do estudo, foram identificadas variáveis
relacionadas à identificação das escolas, situação de funcionamento,
localização, infraestrutura e oferta dos anos iniciais do Ensino Fundamental.

Nesta etapa, verificamos a disponibilidade dessas variáveis nos três
anos. A seleção definitiva dependerá da análise de qualidade dos dados
e da avaliação de sua relevância para o problema estudado.

**Nota metodológica:** A ausência de algumas variáveis na Tabela Escola 
de 2025 não representa indisponibilidade dos dados. Nesse ano, o INEP
passou a disponibilizar as informações em tabelas separadas por domínio.
As variáveis de matrículas, docentes e turmas foram identificadas em
seus respectivos arquivos e integradas posteriormente pela chave
`CO_ENTIDADE`.

In [42]:
# Variáveis candidatas à análise

variaveis_candidatas = {
    "Identificação": [
        "NU_ANO_CENSO",
        "CO_MUNICIPIO",
        "CO_ENTIDADE"
    ],
    "Filtros": [
        "TP_SITUACAO_FUNCIONAMENTO",
        "TP_LOCALIZACAO"
    ],
    "Infraestrutura": [
        "IN_INTERNET",
        "IN_BIBLIOTECA",
        "IN_SALA_LEITURA"
    ],
    "Matrículas": [
        "QT_MAT_FUND_AI",
        "QT_MAT_FUND_AI_INT"
    ],
    "Recursos educacionais": [
        "QT_DOC_FUND_AI",
        "QT_TUR_FUND_AI"
    ]
}

resultado = []

for grupo, variaveis in variaveis_candidatas.items():
    for variavel in variaveis:
        registro = {
            "Grupo": grupo,
            "Variável": variavel
        }

        for ano in sorted(colunas_por_ano):
            registro[str(ano)] = (
                variavel in colunas_por_ano[ano]
            )

        resultado.append(registro)

df_compatibilidade = pd.DataFrame(resultado)

display(df_compatibilidade)

,Grupo,Variável,2022,2023,2024,2025
0,Identificação,NU_ANO_CENSO,True,True,True,True
1,Identificação,CO_MUNICIPIO,True,True,True,True
2,Identificação,CO_ENTIDADE,True,True,True,True
3,Filtros,TP_SITUACAO_FUNCIONAMENTO,True,True,True,True
4,Filtros,TP_LOCALIZACAO,True,True,True,True
5,Infraestrutura,IN_INTERNET,True,True,True,True
6,Infraestrutura,IN_BIBLIOTECA,True,True,True,True
7,Infraestrutura,IN_SALA_LEITURA,True,True,True,True
8,Matrículas,QT_MAT_FUND_AI,True,True,True,False
9,Matrículas,QT_MAT_FUND_AI_INT,True,True,True,False


## 3. Análise exploratória dos registros

Nesta etapa, analisamos o volume de registros, a quantidade
de escolas distintas e a distribuição das situações de
funcionamento.

A análise é realizada antes da aplicação dos filtros,
permitindo avaliar a população original e justificar
posteriormente os critérios de seleção.

In [43]:
# Análise inicial dos registros, um ano por vez

arquivos_escolas = [
    DADOS_DIR / "2022_microdados_ed_basica.csv",
    DADOS_DIR / "2023_microdados_ed_basica.csv",
    DADOS_DIR / "2024_microdados_ed_basica.csv",
    DADOS_DIR / "2025_Tabela_Escola_V2.csv"
]

for arquivo in arquivos_escolas:
    ano = int(arquivo.name[:4])

    df = pd.read_csv(
        arquivo,
        sep=";",
        encoding="latin1",
        usecols=[
            "NU_ANO_CENSO",
            "CO_ENTIDADE",
            "TP_SITUACAO_FUNCIONAMENTO"
        ],
        low_memory=False
    )

    print(f"\n{'=' * 50}")
    print(f"CENSO ESCOLAR {ano}")
    print(f"{'=' * 50}")

    print(f"Total de registros: {len(df):,}")
    print(f"Escolas distintas: {df['CO_ENTIDADE'].nunique():,}")

    print("\nSituação de funcionamento:")
    print(
        df["TP_SITUACAO_FUNCIONAMENTO"]
        .value_counts(dropna=False)
        .sort_index()
    )

    print(
        "\nRegistros com ano divergente:",
        df["NU_ANO_CENSO"].ne(ano).sum()
    )

    del df


CENSO ESCOLAR 2022
Total de registros: 224,649
Escolas distintas: 224,649

Situação de funcionamento:
TP_SITUACAO_FUNCIONAMENTO
1    184332
2     36526
3      3791
Name: count, dtype: int64

Registros com ano divergente: 0

CENSO ESCOLAR 2023
Total de registros: 217,625
Escolas distintas: 217,625

Situação de funcionamento:
TP_SITUACAO_FUNCIONAMENTO
1    180230
2     32083
3      5312
Name: count, dtype: int64

Registros com ano divergente: 0

CENSO ESCOLAR 2024
Total de registros: 215,545
Escolas distintas: 215,545

Situação de funcionamento:
TP_SITUACAO_FUNCIONAMENTO
1    181065
2     31321
3      3159
Name: count, dtype: int64

Registros com ano divergente: 0

CENSO ESCOLAR 2025
Total de registros: 214,192
Escolas distintas: 214,192

Situação de funcionamento:
TP_SITUACAO_FUNCIONAMENTO
1    180540
2     29780
3      3872
Name: count, dtype: int64

Registros com ano divergente: 0


### Considerações sobre a situação de funcionamento

A análise identificou registros de escolas em atividade,
paralisadas e extintas nos três anos avaliados.

Para a construção dos indicadores educacionais, será adotado
o critério de selecionar escolas em atividade
(`TP_SITUACAO_FUNCIONAMENTO == 1`).

Esse critério busca evitar que escolas sem funcionamento
regular no ano de referência influenciem os indicadores
de infraestrutura, matrículas e recursos educacionais.

Os registros originais permanecem preservados, permitindo
a reprodução e a auditoria da seleção.

## 4. Análise da oferta dos anos iniciais do Ensino Fundamental

### Justificativa da seleção

O estudo busca investigar fatores educacionais que possam estar relacionados aos resultados de alfabetização no Brasil.

Considerando a política nacional de alfabetização, que estabelece como referência a alfabetização das crianças até o final do 2º ano do Ensino Fundamental, optou-se por priorizar escolas que oferecem os anos iniciais dessa etapa de ensino.

A variável `QT_MAT_FUND_AI` representa o quantitativo de matrículas nos anos iniciais do Ensino Fundamental (1º ao 5º ano).

Embora essa variável não permita identificar exclusivamente os alunos do 1º e 2º anos, ela possibilita selecionar escolas inseridas na etapa educacional relacionada ao objeto do estudo.

### Critérios de seleção

Serão avaliados dois critérios:

1. Escolas em atividade: `TP_SITUACAO_FUNCIONAMENTO == 1`.
2. Escolas com matrículas nos anos iniciais: `QT_MAT_FUND_AI > 0`.

A análise apresentará o impacto quantitativo desses critérios antes de sua aplicação definitiva.

**Limitação metodológica:** a seleção contempla escolas com matrículas do 1º ao 5º ano, não sendo possível afirmar, apenas com essa variável, que todas possuem alunos matriculados especificamente no 2º ano.

In [44]:
# Análise dos critérios de seleção das escolas

resultados_filtros = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "TP_SITUACAO_FUNCIONAMENTO",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

    else:
        # 2025: situação de funcionamento na tabela Escola
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "TP_SITUACAO_FUNCIONAMENTO"
            ],
            low_memory=False
        )

        # 2025: matrículas na tabela Matrícula
        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

        # Integração das informações por escola
        df = df_escolas.merge(
            df_matriculas,
            on="CO_ENTIDADE",
            how="left",
            validate="one_to_one"
        )

        del df_escolas, df_matriculas

    # Critérios avaliados separadamente
    em_atividade = df["TP_SITUACAO_FUNCIONAMENTO"] == 1
    com_anos_iniciais = df["QT_MAT_FUND_AI"] > 0

    total = len(df)
    ativas = int(em_atividade.sum())
    ativas_com_ai = int((em_atividade & com_anos_iniciais).sum())

    resultados_filtros.append({
        "Ano": ano,
        "Total de escolas": total,
        "Escolas em atividade": ativas,
        "Ativas com anos iniciais": ativas_com_ai,
        "Ativas sem anos iniciais": ativas - ativas_com_ai,
        "Excluídas pelos critérios": total - ativas_com_ai,
        "% selecionadas": round(ativas_com_ai / total * 100, 2)
    })

    del df

df_filtros = pd.DataFrame(resultados_filtros)

display(df_filtros)

,Ano,Total de escolas,Escolas em atividade,Ativas com anos iniciais,Ativas sem anos iniciais,Excluídas pelos critérios,% selecionadas
0,2022,224649,184332,105360,78972,119289,46.90
1,2023,217625,180230,103785,76445,113840,47.69
2,2024,215545,181065,103114,77951,112431,47.84
3,2025,214192,180540,101292,79248,112900,47.29


### Resultados e justificativa dos critérios de seleção

A aplicação conjunta dos critérios de funcionamento e existência de matrículas nos anos iniciais identificou:

- **2022:** 105.360 escolas (46,90% do total).
- **2023:** 103.785 escolas (47,69% do total).
- **2024:** 103.114 escolas (47,84% do total).
- **2025:** 101.292 escolas (47,29% do total).

Os resultados demonstram uma proporção relativamente estável de escolas selecionadas nos quatro anos, variando entre 46,90% e 47,84%.

A exclusão dos demais registros não decorre necessariamente de problemas de qualidade dos dados, mas da delimitação da população de interesse, considerando o objetivo do estudo sobre alfabetização.

A seleção contempla escolas com matrículas nos anos iniciais do Ensino Fundamental, sem restringir a análise exclusivamente ao 2º ano.

**Decisão metodológica:** manter os dois critérios de seleção para as análises subsequentes, preservando os dados originais para rastreabilidade.

## 5. Análise de qualidade dos dados

### 5.1 Valores ausentes

Nesta etapa, avaliamos a presença de valores ausentes nas
variáveis candidatas, considerando exclusivamente escolas
em atividade e com matrículas nos anos iniciais do
Ensino Fundamental.

A análise busca identificar possíveis limitações de
completude dos dados, sem realizar substituições ou
exclusões automáticas.

Os valores ausentes serão posteriormente avaliados
conforme o significado e as regras de preenchimento
das variáveis descritas nos dicionários do Censo Escolar.

In [45]:
# Análise de valores ausentes nas variáveis candidatas

colunas_analise = list(dict.fromkeys(
    variavel
    for grupo in variaveis_candidatas.values()
    for variavel in grupo
))

resultados_nulos = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Estrutura original: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=colunas_analise,
            low_memory=False
        )

    else:
        # Estrutura de 2025: tabelas separadas
        arquivos_2025 = {
            "Escola": "2025_Tabela_Escola_V2.csv",
            "Matrícula": "2025_Tabela_Matricula_V2.csv",
            "Turma": "2025_Tabela_Turma_V2.csv",
            "Docente": "2025_Tabela_Docente_V2.csv"
        }

        colunas_2025 = {
            "Escola": [
                "NU_ANO_CENSO",
                "CO_ENTIDADE",
                "CO_MUNICIPIO",
                "TP_SITUACAO_FUNCIONAMENTO",
                "TP_LOCALIZACAO",
                "IN_INTERNET",
                "IN_BIBLIOTECA",
                "IN_SALA_LEITURA"
            ],
            "Matrícula": [
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI",
                "QT_MAT_FUND_AI_INT"
            ],
            "Turma": [
                "CO_ENTIDADE",
                "QT_TUR_FUND_AI"
            ],
            "Docente": [
                "CO_ENTIDADE",
                "QT_DOC_FUND_AI"
            ]
        }

        tabelas_2025 = {}

        for nome, arquivo in arquivos_2025.items():
            tabelas_2025[nome] = pd.read_csv(
                DADOS_DIR / arquivo,
                sep=";",
                encoding="latin1",
                usecols=colunas_2025[nome],
                low_memory=False
            )

        # Integração por escola
        df = tabelas_2025["Escola"].copy()

        for nome in ["Matrícula", "Turma", "Docente"]:
            df = df.merge(
                tabelas_2025[nome],
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )

        del tabelas_2025

    # Aplica os critérios de seleção já justificados
    df_filtrado = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ]

    total = len(df_filtrado)

    for coluna in colunas_analise:
        nulos = int(df_filtrado[coluna].isna().sum())

        resultados_nulos.append({
            "Ano": ano,
            "Variável": coluna,
            "Registros": total,
            "Nulos": nulos,
            "% Nulos": round(nulos / total * 100, 2)
        })

    del df, df_filtrado

df_nulos = pd.DataFrame(resultados_nulos)

# Exibe somente variáveis com valores ausentes
display(
    df_nulos[df_nulos["Nulos"] > 0]
    .sort_values(["Ano", "% Nulos"], ascending=[True, False])
)

,Ano,Variável,Registros,Nulos,% Nulos
10,2022,QT_DOC_FUND_AI,105360,3,0.0


In [46]:
# Investigação dos registros com docentes não informados em 2022

arquivo_2022 = next(
    arquivo for arquivo in arquivos
    if arquivo.name.startswith("2022")
)

df_2022 = pd.read_csv(
    arquivo_2022,
    sep=";",
    encoding="latin1",
    usecols=colunas_analise,
    low_memory=False
)

# Mantém a população de interesse
df_2022 = df_2022.loc[
    (df_2022["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
    (df_2022["QT_MAT_FUND_AI"] > 0)
]

# Identifica os registros com valores ausentes
df_investigacao = df_2022.loc[
    df_2022["QT_DOC_FUND_AI"].isna(),
    [
        "CO_MUNICIPIO",
        "CO_ENTIDADE",
        "QT_MAT_FUND_AI",
        "QT_DOC_FUND_AI",
        "QT_TUR_FUND_AI"
    ]
]

display(df_investigacao)

del df_2022

,CO_MUNICIPIO,CO_ENTIDADE,QT_MAT_FUND_AI,QT_DOC_FUND_AI,QT_TUR_FUND_AI
97308,2922300,29478391,12.0,NaN,1.0
100014,2927408,29180848,118.0,NaN,6.0
101976,2927408,29466040,11.0,NaN,4.0


### 5.2 Investigação e tratamento dos valores ausentes

A análise identificou três escolas em 2022 com valores ausentes na variável `QT_DOC_FUND_AI`.

A investigação demonstrou que todas possuem matrículas e turmas registradas nos anos iniciais do Ensino Fundamental, não sendo possível concluir que a ausência da informação representa inexistência de docentes.

Considerando a baixa incidência de valores ausentes (0,00285% das escolas selecionadas em 2022), optou-se por:

- Preservar os três registros na base de dados.
- Manter os valores ausentes, sem substituição automática por zero.
- Desconsiderar esses valores apenas nos cálculos que dependam da quantidade de docentes, conforme a metodologia de cada indicador.

**Decisão metodológica:** preservar a integridade dos dados originais e evitar imputações sem fundamentação.

### 5.3 Consistência das variáveis quantitativas

Nesta etapa, avaliamos a consistência dos quantitativos de
matrículas, matrículas em tempo integral, docentes e turmas
nos anos iniciais do Ensino Fundamental.

Serão investigados:
- Valores negativos.
- Quantidades zeradas.
- Matrículas em tempo integral superiores ao total de matrículas.
- Escolas com matrículas, mas sem docentes ou turmas informados.

As ocorrências identificadas serão tratadas como alertas
de consistência, não necessariamente como erros.

Nenhuma correção ou exclusão será realizada automaticamente.

In [47]:
# Análise de consistência das variáveis quantitativas

variaveis_numericas = [
    "QT_MAT_FUND_AI",
    "QT_MAT_FUND_AI_INT",
    "QT_DOC_FUND_AI",
    "QT_TUR_FUND_AI"
]

resultados_consistencia = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "TP_SITUACAO_FUNCIONAMENTO",
                *variaveis_numericas
            ],
            low_memory=False
        )

    else:
        # Em 2025: informações distribuídas em quatro tabelas
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "TP_SITUACAO_FUNCIONAMENTO"
            ],
            low_memory=False
        )

        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI",
                "QT_MAT_FUND_AI_INT"
            ],
            low_memory=False
        )

        df_docentes = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Docente_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_DOC_FUND_AI"
            ],
            low_memory=False
        )

        df_turmas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Turma_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

        # Integração por escola
        df = (
            df_escolas
            .merge(
                df_matriculas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_docentes,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_turmas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
        )

        del df_escolas, df_matriculas, df_docentes, df_turmas

    # População selecionada
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ]

    resultado = {
        "Ano": ano,
        "Escolas": len(df),

        "Valores negativos": int(
            (df[variaveis_numericas] < 0).sum().sum()
        ),

        "Matrículas integrais > total": int(
            (
                df["QT_MAT_FUND_AI_INT"] >
                df["QT_MAT_FUND_AI"]
            ).sum()
        ),

        "Docentes zerados": int(
            (df["QT_DOC_FUND_AI"] == 0).sum()
        ),

        "Turmas zeradas": int(
            (df["QT_TUR_FUND_AI"] == 0).sum()
        ),

        "Docentes ausentes": int(
            df["QT_DOC_FUND_AI"].isna().sum()
        ),

        "Turmas ausentes": int(
            df["QT_TUR_FUND_AI"].isna().sum()
        )
    }

    resultados_consistencia.append(resultado)

    del df

df_consistencia = pd.DataFrame(resultados_consistencia)

display(df_consistencia)

,Ano,Escolas,Valores negativos,Matrículas integrais > total,Docentes zerados,Turmas zeradas,Docentes ausentes,Turmas ausentes
0,2022,105360,0,0,14089,14078,3,0
1,2023,103785,0,0,14170,14166,0,0
2,2024,103114,0,0,13784,13784,0,0
3,2025,101292,0,0,13399,13399,0,0


In [48]:
# Investigação de escolas com docentes e turmas zerados

resultados_zeros = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "TP_SITUACAO_FUNCIONAMENTO",
                "QT_MAT_FUND_AI",
                "QT_DOC_FUND_AI",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

    else:
        # 2025: integração das tabelas por escola
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "TP_SITUACAO_FUNCIONAMENTO"
            ],
            low_memory=False
        )

        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

        df_docentes = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Docente_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_DOC_FUND_AI"
            ],
            low_memory=False
        )

        df_turmas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Turma_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

        df = (
            df_escolas
            .merge(
                df_matriculas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_docentes,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_turmas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
        )

        del df_escolas, df_matriculas, df_docentes, df_turmas

    # População selecionada
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ]

    docentes_zero = df["QT_DOC_FUND_AI"] == 0
    turmas_zero = df["QT_TUR_FUND_AI"] == 0

    resultados_zeros.append({
        "Ano": ano,

        "Docentes e turmas zerados": int(
            (docentes_zero & turmas_zero).sum()
        ),

        "Somente docentes zerados": int(
            (docentes_zero & ~turmas_zero).sum()
        ),

        "Somente turmas zeradas": int(
            (~docentes_zero & turmas_zero).sum()
        ),

        "Matrículas em escolas com turmas zeradas": int(
            df.loc[turmas_zero, "QT_MAT_FUND_AI"].sum()
        ),

        "Mediana de matrículas nessas escolas": float(
            df.loc[turmas_zero, "QT_MAT_FUND_AI"].median()
        )
    })

    del df

df_zeros = pd.DataFrame(resultados_zeros)

display(df_zeros)

,Ano,Docentes e turmas zerados,Somente docentes zerados,Somente turmas zeradas,Matrículas em escolas com turmas zeradas,Mediana de matrículas nessas escolas
0,2022,14078,11,0,302078,19.0
1,2023,14166,4,0,297447,18.0
2,2024,13784,0,0,287051,18.0
3,2025,13399,0,0,273671,18.0


In [49]:
# Perfil das escolas com turmas zeradas em 2023

arquivo_2023 = next(
    a for a in arquivos if a.name.startswith("2023")
)

df = pd.read_csv(
    arquivo_2023,
    sep=";",
    encoding="latin1",
    usecols=[
        "TP_SITUACAO_FUNCIONAMENTO",
        "TP_LOCALIZACAO",
        "TP_DEPENDENCIA",
        "QT_MAT_FUND_AI",
        "QT_DOC_FUND_AI",
        "QT_TUR_FUND_AI"
    ],
    low_memory=False
)

df = df.loc[
    (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
    (df["QT_MAT_FUND_AI"] > 0)
]

df["turmas_zeradas"] = df["QT_TUR_FUND_AI"] == 0

for coluna in ["TP_LOCALIZACAO", "TP_DEPENDENCIA"]:
    print(f"\nDistribuição por {coluna}")

    display(
        pd.crosstab(
            df[coluna],
            df["turmas_zeradas"],
            margins=True
        )
    )

del df


Distribuição por TP_LOCALIZACAO


turmas_zeradas,False,True,All
TP_LOCALIZACAO,,,
1,59474,787,60261
2,30145,13379,43524
All,89619,14166,103785



Distribuição por TP_DEPENDENCIA


turmas_zeradas,False,True,All
TP_DEPENDENCIA,,,
1,26,0,26
2,7634,1401,9035
3,58583,12303,70886
4,23376,462,23838
All,89619,14166,103785


In [50]:
# Comparação de turmas zeradas por localização e ano

resultados_localizacao = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "TP_SITUACAO_FUNCIONAMENTO",
                "TP_LOCALIZACAO",
                "QT_MAT_FUND_AI",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

    else:
        # 2025: informações distribuídas em três tabelas
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "TP_SITUACAO_FUNCIONAMENTO",
                "TP_LOCALIZACAO"
            ],
            low_memory=False
        )

        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

        df_turmas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Turma_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

        # Integração por escola
        df = (
            df_escolas
            .merge(
                df_matriculas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_turmas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
        )

        del df_escolas, df_matriculas, df_turmas

    # População selecionada
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ]

    resumo = (
        df.groupby("TP_LOCALIZACAO")
        .agg(
            escolas=("QT_TUR_FUND_AI", "size"),
            turmas_zeradas=(
                "QT_TUR_FUND_AI",
                lambda x: (x == 0).sum()
            )
        )
        .reset_index()
    )

    resumo["Ano"] = ano

    resumo["% turmas zeradas"] = (
        resumo["turmas_zeradas"] /
        resumo["escolas"] * 100
    ).round(2)

    resultados_localizacao.append(resumo)

    del df

df_localizacao = pd.concat(
    resultados_localizacao,
    ignore_index=True
)

display(
    df_localizacao[
        [
            "Ano",
            "TP_LOCALIZACAO",
            "escolas",
            "turmas_zeradas",
            "% turmas zeradas"
        ]
    ]
)

,Ano,TP_LOCALIZACAO,escolas,turmas_zeradas,% turmas zeradas
0,2022,1,60537,861,1.42
1,2022,2,44823,13217,29.49
2,2023,1,60261,787,1.31
3,2023,2,43524,13379,30.74
4,2024,1,60307,766,1.27
5,2024,2,42807,13018,30.41
6,2025,1,59771,814,1.36
7,2025,2,41521,12585,30.31


### 5.4 Investigação dos quantitativos zerados

A análise identificou escolas com matrículas nos anos iniciais do Ensino Fundamental, mas com quantitativos de turmas e docentes iguais a zero.

A investigação demonstrou uma concentração desse comportamento nas escolas rurais, com percentuais entre 29,49% e 30,74% nos anos de 2022 a 2024. Nas escolas urbanas, os percentuais permaneceram entre 1,27% e 1,42%.

A persistência desse padrão sugere que os valores zerados podem estar relacionados a características específicas dos registros ou da organização educacional, embora a análise não permita determinar sua causa.

**Decisão metodológica:**
- Preservar os registros e os valores originais.
- Não substituir automaticamente valores zerados.
- Evitar divisões por zero na construção de indicadores.
- Documentar limitações de cobertura dos indicadores derivados de docentes e turmas.

Essa abordagem busca evitar exclusões potencialmente enviesadas, especialmente de escolas rurais.

### 5.5 Análise das variáveis de infraestrutura

Nesta etapa, avaliamos as variáveis relacionadas à infraestrutura
das escolas:

- IN_INTERNET: acesso à internet.
- IN_BIBLIOTECA: existência de biblioteca.
- IN_SALA_LEITURA: existência de sala de leitura.

A análise busca identificar:
- Distribuição dos valores em cada ano.
- Presença de valores ausentes ou inesperados.
- Percentual de escolas com cada recurso disponível.

A avaliação considera somente escolas em funcionamento e
com matrículas nos anos iniciais do Ensino Fundamental.

A presença desses recursos será analisada como possível
característica explicativa, sem pressupor relação causal
com os resultados de alfabetização.

In [51]:
# Distribuição das variáveis de infraestrutura

variaveis_infraestrutura = [
    "IN_INTERNET",
    "IN_BIBLIOTECA",
    "IN_SALA_LEITURA"
]

resultados_infraestrutura = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "TP_SITUACAO_FUNCIONAMENTO",
                "QT_MAT_FUND_AI",
                *variaveis_infraestrutura
            ],
            low_memory=False
        )

    else:
        # 2025: infraestrutura na tabela Escola
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "TP_SITUACAO_FUNCIONAMENTO",
                *variaveis_infraestrutura
            ],
            low_memory=False
        )

        # 2025: matrículas na tabela Matrícula
        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

        # Integração por escola
        df = df_escolas.merge(
            df_matriculas,
            on="CO_ENTIDADE",
            how="left",
            validate="one_to_one"
        )

        del df_escolas, df_matriculas

    # Aplica os critérios de seleção
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ]

    for variavel in variaveis_infraestrutura:
        contagem = df[variavel].value_counts(dropna=False)

        resultados_infraestrutura.append({
            "Ano": ano,
            "Variável": variavel,
            "Escolas": len(df),
            "Valor 1": int(contagem.get(1, 0)),
            "Valor 0": int(contagem.get(0, 0)),
            "Nulos": int(df[variavel].isna().sum()),
            "Outros valores": int(
                (
                    ~df[variavel].isin([0, 1]) &
                    df[variavel].notna()
                ).sum()
            ),
            "% Valor 1": round(
                (df[variavel] == 1).mean() * 100, 2
            )
        })

    del df

df_infraestrutura = pd.DataFrame(resultados_infraestrutura)

display(df_infraestrutura)

,Ano,Variável,Escolas,Valor 1,Valor 0,Nulos,Outros valores,% Valor 1
0,2022,IN_INTERNET,105360,87211,18149,0,0,82.77
1,2022,IN_BIBLIOTECA,105360,39025,66335,0,0,37.04
2,2022,IN_SALA_LEITURA,105360,29242,76118,0,0,27.75
3,2023,IN_INTERNET,103785,90454,13331,0,0,87.16
4,2023,IN_BIBLIOTECA,103785,38727,65058,0,0,37.31
5,2023,IN_SALA_LEITURA,103785,29739,74046,0,0,28.65
6,2024,IN_INTERNET,103114,91670,11444,0,0,88.90
7,2024,IN_BIBLIOTECA,103114,38810,64304,0,0,37.64
8,2024,IN_SALA_LEITURA,103114,31159,71955,0,0,30.22
9,2025,IN_INTERNET,101292,93245,8047,0,0,92.06


### 5.6 Infraestrutura por localização

A análise compara a disponibilidade de internet, biblioteca e
sala de leitura entre escolas urbanas e rurais.

O objetivo é identificar possíveis desigualdades de
infraestrutura e avaliar a relevância da localização
na interpretação dessas características.

Os percentuais são calculados dentro de cada grupo de
localização, considerando apenas escolas em funcionamento
com matrículas nos anos iniciais do Ensino Fundamental.

In [52]:
# Infraestrutura por localização e ano

resultados_infra_localizacao = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "TP_SITUACAO_FUNCIONAMENTO",
                "TP_LOCALIZACAO",
                "QT_MAT_FUND_AI",
                *variaveis_infraestrutura
            ],
            low_memory=False
        )

    else:
        # 2025: informações distribuídas em duas tabelas
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "TP_SITUACAO_FUNCIONAMENTO",
                "TP_LOCALIZACAO",
                *variaveis_infraestrutura
            ],
            low_memory=False
        )

        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

        # Integração por escola
        df = df_escolas.merge(
            df_matriculas,
            on="CO_ENTIDADE",
            how="left",
            validate="one_to_one"
        )

        del df_escolas, df_matriculas

    # Seleciona escolas ativas com matrículas nos anos iniciais
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ]

    # Consolida os indicadores por localização
    resumo = (
        df.groupby("TP_LOCALIZACAO")
        .agg(
            Escolas=("QT_MAT_FUND_AI", "size"),
            Internet=("IN_INTERNET", "mean"),
            Biblioteca=("IN_BIBLIOTECA", "mean"),
            Sala_leitura=("IN_SALA_LEITURA", "mean")
        )
        .reset_index()
    )

    resumo["Ano"] = ano

    for coluna in ["Internet", "Biblioteca", "Sala_leitura"]:
        resumo[coluna] = (resumo[coluna] * 100).round(2)

    resultados_infra_localizacao.append(resumo)

    del df

df_infra_localizacao = pd.concat(
    resultados_infra_localizacao,
    ignore_index=True
)

df_infra_localizacao["Localização"] = (
    df_infra_localizacao["TP_LOCALIZACAO"].map({
        1: "Urbana",
        2: "Rural"
    })
)

display(
    df_infra_localizacao[
        [
            "Ano",
            "Localização",
            "Escolas",
            "Internet",
            "Biblioteca",
            "Sala_leitura"
        ]
    ]
)

,Ano,Localização,Escolas,Internet,Biblioteca,Sala_leitura
0,2022,Urbana,60537,98.11,53.69,40.07
1,2022,Rural,44823,62.06,14.56,11.12
2,2023,Urbana,60261,98.82,53.48,40.70
3,2023,Rural,43524,71.00,14.94,11.97
4,2024,Urbana,60307,99.01,53.46,42.09
5,2024,Rural,42807,74.67,15.34,13.49
6,2025,Urbana,59771,99.42,53.33,45.51
7,2025,Rural,41521,81.45,16.03,16.71


### 5.7 Resultados da análise de infraestrutura

As três variáveis de infraestrutura apresentaram valores binários
válidos, sem registros ausentes, nos anos de 2022 a 2024.

A análise identificou diferenças expressivas entre escolas
urbanas e rurais.

A disponibilidade de internet nas escolas rurais aumentou de
62,06% em 2022 para 74,67% em 2024, enquanto nas urbanas
passou de 98,11% para 99,01%.

As diferenças na disponibilidade de bibliotecas e salas de
leitura permaneceram elevadas. Em 2024, bibliotecas estavam
presentes em 53,46% das escolas urbanas e 15,34% das rurais.

**Decisão metodológica:**

- Manter as três variáveis de infraestrutura como candidatas.
- Preservar a localização urbana/rural para análises e agregações.
- Não interpretar as associações descritivas como relações causais.
- Avaliar posteriormente a relação dessas características com
  os resultados municipais de alfabetização.

### 5.8 Disponibilidade de espaços de leitura

As variáveis IN_BIBLIOTECA e IN_SALA_LEITURA representam
recursos distintos, mas relacionados à infraestrutura
de apoio à leitura.

Nesta etapa, analisamos a combinação dessas características
para identificar escolas que possuem:

- Nenhum dos recursos.
- Somente biblioteca.
- Somente sala de leitura.
- Ambos os recursos.

O objetivo é avaliar a possibilidade de construir um
indicador derivado que represente a disponibilidade de
pelo menos um espaço de leitura.

A criação desse indicador não implica substituir as
variáveis originais.

In [53]:
# Combinação de biblioteca e sala de leitura

import numpy as np

resultados_espacos_leitura = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "TP_SITUACAO_FUNCIONAMENTO",
                "TP_LOCALIZACAO",
                "QT_MAT_FUND_AI",
                "IN_BIBLIOTECA",
                "IN_SALA_LEITURA"
            ],
            low_memory=False
        )

    else:
        # 2025: dados de infraestrutura na tabela Escola
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "TP_SITUACAO_FUNCIONAMENTO",
                "TP_LOCALIZACAO",
                "IN_BIBLIOTECA",
                "IN_SALA_LEITURA"
            ],
            low_memory=False
        )

        # Matrículas dos anos iniciais
        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

        # Integração por escola
        df = df_escolas.merge(
            df_matriculas,
            on="CO_ENTIDADE",
            how="left",
            validate="one_to_one"
        )

        del df_escolas, df_matriculas

    # Seleciona escolas ativas com matrículas nos anos iniciais
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ].copy()

    # Classificação das combinações
    condicoes = [
        (df["IN_BIBLIOTECA"] == 0) &
        (df["IN_SALA_LEITURA"] == 0),

        (df["IN_BIBLIOTECA"] == 1) &
        (df["IN_SALA_LEITURA"] == 0),

        (df["IN_BIBLIOTECA"] == 0) &
        (df["IN_SALA_LEITURA"] == 1),

        (df["IN_BIBLIOTECA"] == 1) &
        (df["IN_SALA_LEITURA"] == 1)
    ]

    categorias = [
        "Nenhum recurso",
        "Somente biblioteca",
        "Somente sala de leitura",
        "Ambos"
    ]

    df["espaco_leitura"] = np.select(
        condicoes,
        categorias,
        default="Não classificado"
    )

    resumo = (
        df.groupby(
            ["TP_LOCALIZACAO", "espaco_leitura"]
        )
        .size()
        .reset_index(name="Escolas")
    )

    resumo["Ano"] = ano

    resumo["% do grupo"] = (
        resumo["Escolas"] /
        resumo.groupby("TP_LOCALIZACAO")["Escolas"]
        .transform("sum") * 100
    ).round(2)

    resultados_espacos_leitura.append(resumo)

    del df

df_espacos_leitura = pd.concat(
    resultados_espacos_leitura,
    ignore_index=True
)

df_espacos_leitura["Localização"] = (
    df_espacos_leitura["TP_LOCALIZACAO"].map({
        1: "Urbana",
        2: "Rural"
    })
)

display(
    df_espacos_leitura[
        [
            "Ano",
            "Localização",
            "espaco_leitura",
            "Escolas",
            "% do grupo"
        ]
    ].sort_values(
        ["Ano", "Localização", "espaco_leitura"]
    )
)

,Ano,Localização,espaco_leitura,Escolas,% do grupo
4,2022,Rural,Ambos,1176,2.62
5,2022,Rural,Nenhum recurso,34490,76.95
6,2022,Rural,Somente biblioteca,5348,11.93
7,2022,Rural,Somente sala de leitura,3809,8.50
0,2022,Urbana,Ambos,11692,19.31
1,2022,Urbana,Nenhum recurso,15471,25.56
2,2022,Urbana,Somente biblioteca,20809,34.37
3,2022,Urbana,Somente sala de leitura,12565,20.76
12,2023,Rural,Ambos,1280,2.94
13,2023,Rural,Nenhum recurso,33092,76.03


### 5.9 Conclusão da análise de infraestrutura

A análise conjunta de biblioteca e sala de leitura demonstrou
que esses recursos são complementares na caracterização
da infraestrutura escolar.

Em 2024, 74,59% das escolas rurais selecionadas não possuíam
nenhum dos dois recursos, contra 24,81% das escolas urbanas.

Também foram identificadas escolas que possuíam exclusivamente
sala de leitura, demonstrando que considerar apenas bibliotecas
não representaria toda a disponibilidade desses espaços.

**Decisão metodológica:**

- Manter IN_INTERNET, IN_BIBLIOTECA e IN_SALA_LEITURA.
- Criar IN_ESPACO_LEITURA, indicando a presença de biblioteca
  ou sala de leitura.
- Preservar TP_LOCALIZACAO para análises de desigualdade.
- Avaliar posteriormente a associação dessas características
  com os resultados municipais de alfabetização.

A variável derivada representa disponibilidade declarada
de infraestrutura, não sua qualidade, utilização ou impacto
sobre a aprendizagem.

### 6. Análise dos indicadores municipais

Após avaliar a qualidade e a consistência dos registros
escolares, iniciamos a investigação das agregações municipais.

O objetivo é avaliar a viabilidade de construir indicadores
relacionados a matrículas, docentes e turmas dos anos iniciais
do Ensino Fundamental.

Nesta etapa, serão investigados:

- Quantidade de escolas por município.
- Total de matrículas por município.
- Total de docentes e turmas informados.
- Municípios com quantitativos zerados.
- Participação de escolas com turmas zeradas nas matrículas.

As agregações serão realizadas separadamente por ano.

Os resultados serão utilizados para definir os indicadores
candidatos à camada Gold, sem realizar tratamentos
automáticos dos valores zerados.

In [54]:
# Análise da cobertura dos indicadores municipais

resultados_municipais = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_MUNICIPIO",
                "TP_SITUACAO_FUNCIONAMENTO",
                "QT_MAT_FUND_AI",
                "QT_DOC_FUND_AI",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

    else:
        # 2025: informações distribuídas em quatro tabelas
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "CO_MUNICIPIO",
                "TP_SITUACAO_FUNCIONAMENTO"
            ],
            low_memory=False
        )

        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

        df_docentes = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Docente_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_DOC_FUND_AI"
            ],
            low_memory=False
        )

        df_turmas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Turma_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

        # Integração por escola
        df = (
            df_escolas
            .merge(
                df_matriculas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_docentes,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_turmas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
        )

        del df_escolas, df_matriculas, df_docentes, df_turmas

    # Seleciona escolas ativas com matrículas nos anos iniciais
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ].copy()

    # Identifica escolas com turmas zeradas
    df["escola_turmas_zero"] = (
        df["QT_TUR_FUND_AI"] == 0
    )

    # Matrículas vinculadas a escolas com turmas zeradas
    df["matriculas_turmas_zero"] = (
        df["QT_MAT_FUND_AI"].where(
            df["escola_turmas_zero"], 0
        )
    )

    # Consolidação municipal
    municipal = (
        df.groupby("CO_MUNICIPIO")
        .agg(
            escolas=("QT_MAT_FUND_AI", "size"),
            matriculas=("QT_MAT_FUND_AI", "sum"),
            docentes=("QT_DOC_FUND_AI", "sum"),
            turmas=("QT_TUR_FUND_AI", "sum"),
            escolas_turmas_zero=(
                "escola_turmas_zero", "sum"
            ),
            matriculas_turmas_zero=(
                "matriculas_turmas_zero", "sum"
            )
        )
        .reset_index()
    )

    resultados_municipais.append({
        "Ano": ano,
        "Municípios": len(municipal),
        "Municípios sem docentes": int(
            (municipal["docentes"] == 0).sum()
        ),
        "Municípios sem turmas": int(
            (municipal["turmas"] == 0).sum()
        ),
        "Municípios com escolas sem turmas": int(
            (municipal["escolas_turmas_zero"] > 0).sum()
        ),
        "% matrículas em escolas sem turmas": round(
            municipal["matriculas_turmas_zero"].sum() /
            municipal["matriculas"].sum() * 100,
            2
        )
    })

    del df, municipal

df_cobertura_municipal = pd.DataFrame(
    resultados_municipais
)

display(df_cobertura_municipal)

,Ano,Municípios,Municípios sem docentes,Municípios sem turmas,Municípios com escolas sem turmas,% matrículas em escolas sem turmas
0,2022,5570,0,0,2689,2.08
1,2023,5570,0,0,2674,2.06
2,2024,5570,0,0,2663,1.98
3,2025,5571,0,0,2612,1.89


### 6.1 Cobertura dos quantitativos municipais

A agregação identificou totais positivos de docentes e turmas
em todos os municípios analisados nos três anos.

Entretanto, parte das escolas apresenta matrículas positivas
e quantitativos de turmas zerados.

Para avaliar o possível impacto dessa característica nos
indicadores municipais, será calculada a participação das
matrículas dessas escolas no total de matrículas do município.

Essa proporção será utilizada como indicador de atenção
à cobertura dos dados, sem assumir que todos os valores
zerados representem erros ou informações ausentes.

In [55]:
# Distribuição municipal da participação de matrículas
# em escolas com turmas zeradas

resultados_cobertura = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_MUNICIPIO",
                "TP_SITUACAO_FUNCIONAMENTO",
                "QT_MAT_FUND_AI",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

    else:
        # 2025: informações distribuídas em três tabelas
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "CO_MUNICIPIO",
                "TP_SITUACAO_FUNCIONAMENTO"
            ],
            low_memory=False
        )

        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

        df_turmas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Turma_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

        # Integração por escola
        df = (
            df_escolas
            .merge(
                df_matriculas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_turmas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
        )

        del df_escolas, df_matriculas, df_turmas

    # Seleciona escolas ativas com matrículas nos anos iniciais
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ].copy()

    # Matrículas vinculadas a escolas com turmas zeradas
    df["matriculas_turmas_zero"] = (
        df["QT_MAT_FUND_AI"].where(
            df["QT_TUR_FUND_AI"] == 0, 0
        )
    )

    # Consolidação municipal
    municipal = (
        df.groupby("CO_MUNICIPIO")
        .agg(
            matriculas=("QT_MAT_FUND_AI", "sum"),
            matriculas_turmas_zero=(
                "matriculas_turmas_zero", "sum"
            )
        )
    )

    # Participação percentual por município
    municipal["pct_matriculas_turmas_zero"] = (
        municipal["matriculas_turmas_zero"] /
        municipal["matriculas"] * 100
    )

    pct = municipal["pct_matriculas_turmas_zero"]

    resultados_cobertura.append({
        "Ano": ano,
        "Municípios": len(municipal),
        "Sem ocorrência": int((pct == 0).sum()),
        "Até 5%": int(((pct > 0) & (pct <= 5)).sum()),
        "Acima de 5% até 10%": int(
            ((pct > 5) & (pct <= 10)).sum()
        ),
        "Acima de 10% até 20%": int(
            ((pct > 10) & (pct <= 20)).sum()
        ),
        "Acima de 20%": int((pct > 20).sum()),
        "Maior percentual": round(pct.max(), 2)
    })

    del df, municipal

df_cobertura_distribuicao = pd.DataFrame(
    resultados_cobertura
)

display(df_cobertura_distribuicao)

,Ano,Municípios,Sem ocorrência,Até 5%,Acima de 5% até 10%,Acima de 10% até 20%,Acima de 20%,Maior percentual
0,2022,5570,2881,1424,630,439,196,68.11
1,2023,5570,2896,1380,661,449,184,71.48
2,2024,5570,2907,1397,646,434,186,68.59
3,2025,5571,2959,1425,608,407,172,68.42


### 6.2 Avaliação da cobertura municipal

A análise confirmou que todos os 5.570 municípios apresentam
totais positivos de docentes e turmas nos três anos.

Entretanto, a participação de matrículas em escolas com
turmas zeradas varia significativamente entre municípios.

Em 2024, 2.907 municípios não apresentaram ocorrências,
enquanto 620 registraram mais de 10% de suas matrículas
em escolas com turmas zeradas. O maior percentual observado
foi de 68,59%.

**Decisão metodológica:**

- Manter matrículas, docentes e turmas como variáveis candidatas.
- Preservar os valores originais nas agregações.
- Criar a métrica pct_matriculas_escolas_turmas_zero.
- Não excluir municípios automaticamente.
- Considerar a cobertura na interpretação de indicadores
  como alunos por docente e alunos por turma.

A métrica proposta sinaliza possíveis limitações de
cobertura, sem classificar automaticamente os registros
como incorretos.

### 6.3 Distribuição dos indicadores municipais

Nesta etapa, avaliamos dois indicadores candidatos:

- Alunos por docente: total municipal de matrículas dividido
  pelo quantitativo informado de docentes.
- Alunos por turma: total municipal de matrículas dividido
  pelo quantitativo informado de turmas.

A análise busca identificar a distribuição dos indicadores,
seus valores extremos e possíveis associações com a
participação de matrículas em escolas com turmas zeradas.

Os indicadores são calculados separadamente por ano,
considerando escolas em funcionamento com matrículas
nos anos iniciais do Ensino Fundamental.

Os resultados serão tratados como indicadores descritivos,
sem pressupor que representem o tamanho real das turmas
ou a quantidade de alunos atendidos individualmente
por cada docente.

In [56]:
# Distribuição dos indicadores municipais

resultados_indicadores = []
resumos_indicadores = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_MUNICIPIO",
                "TP_SITUACAO_FUNCIONAMENTO",
                "QT_MAT_FUND_AI",
                "QT_DOC_FUND_AI",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

    else:
        # 2025: informações distribuídas em quatro tabelas
        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "CO_MUNICIPIO",
                "TP_SITUACAO_FUNCIONAMENTO"
            ],
            low_memory=False
        )

        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI"
            ],
            low_memory=False
        )

        df_docentes = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Docente_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_DOC_FUND_AI"
            ],
            low_memory=False
        )

        df_turmas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Turma_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

        # Integração por escola
        df = (
            df_escolas
            .merge(
                df_matriculas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_docentes,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_turmas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
        )

        del df_escolas, df_matriculas, df_docentes, df_turmas

    # Seleciona escolas ativas com matrículas nos anos iniciais
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ].copy()

    # Matrículas vinculadas a escolas com turmas zeradas
    df["matriculas_turmas_zero"] = (
        df["QT_MAT_FUND_AI"].where(
            df["QT_TUR_FUND_AI"] == 0, 0
        )
    )

    # Consolidação municipal
    municipal = (
        df.groupby("CO_MUNICIPIO")
        .agg(
            matriculas=("QT_MAT_FUND_AI", "sum"),
            docentes=("QT_DOC_FUND_AI", "sum"),
            turmas=("QT_TUR_FUND_AI", "sum"),
            matriculas_turmas_zero=(
                "matriculas_turmas_zero", "sum"
            ),
            escolas_docentes_nulos=(
                "QT_DOC_FUND_AI",
                lambda x: x.isna().sum()
            )
        )
        .reset_index()
    )

    # Indicadores municipais
    municipal["alunos_por_docente"] = (
        municipal["matriculas"] /
        municipal["docentes"].replace(0, float("nan"))
    )

    municipal["alunos_por_turma"] = (
        municipal["matriculas"] /
        municipal["turmas"].replace(0, float("nan"))
    )

    municipal["pct_matriculas_escolas_turmas_zero"] = (
        municipal["matriculas_turmas_zero"] /
        municipal["matriculas"] * 100
    )

    municipal["Ano"] = ano

    resultados_indicadores.append(municipal)

    # Resumo estatístico dos indicadores
    for indicador in [
        "alunos_por_docente",
        "alunos_por_turma"
    ]:
        valores = municipal[indicador].dropna()

        resumos_indicadores.append({
            "Ano": ano,
            "Indicador": indicador,
            "Municípios válidos": len(valores),
            "Mediana": round(valores.median(), 2),
            "P90": round(valores.quantile(0.90), 2),
            "P95": round(valores.quantile(0.95), 2),
            "Máximo": round(valores.max(), 2)
        })

    del df, municipal

df_indicadores_municipais = pd.concat(
    resultados_indicadores,
    ignore_index=True
)

df_resumo_indicadores = pd.DataFrame(
    resumos_indicadores
)

display(df_resumo_indicadores)

,Ano,Indicador,Municípios válidos,Mediana,P90,P95,Máximo
0,2022,alunos_por_docente,5570,15.43,22.41,24.76,75.42
1,2022,alunos_por_turma,5570,19.58,24.31,26.20,71.65
2,2023,alunos_por_docente,5570,15.29,22.15,24.83,84.53
3,2023,alunos_por_turma,5570,19.58,24.25,26.06,76.48
4,2024,alunos_por_docente,5570,15.10,21.88,24.12,78.42
5,2024,alunos_por_turma,5570,19.61,24.27,25.86,70.95
6,2025,alunos_por_docente,5571,14.71,21.36,23.49,67.23
7,2025,alunos_por_turma,5571,19.54,24.04,25.55,67.23


### 6.4 Investigação dos valores extremos

A distribuição dos indicadores municipais apresentou
medianas e percentis relativamente estáveis entre
2022 e 2025.

Entretanto, foram identificados valores máximos
consideravelmente superiores ao percentil 95.

Nesta etapa, serão investigados os municípios com
maiores razões de alunos por docente e alunos por turma.

A investigação considera também a participação das
matrículas em escolas com turmas zeradas, buscando
identificar possíveis limitações de cobertura.

Valores extremos não serão automaticamente considerados
incorretos ou excluídos.

In [57]:
# Cinco maiores valores de cada indicador por ano

indicadores = [
    "alunos_por_docente",
    "alunos_por_turma"
]

for ano in sorted(df_indicadores_municipais["Ano"].unique()):
    df_ano = df_indicadores_municipais.loc[
        df_indicadores_municipais["Ano"] == ano
    ]

    for indicador in indicadores:
        print(f"\nAno: {ano} | Indicador: {indicador}")

        colunas = [
            "CO_MUNICIPIO",
            "matriculas",
            "docentes",
            "turmas",
            indicador,
            "pct_matriculas_escolas_turmas_zero",
            "escolas_docentes_nulos"
        ]

        display(
            df_ano.nlargest(5, indicador)[colunas].round(2)
        )


Ano: 2022 | Indicador: alunos_por_docente


,CO_MUNICIPIO,matriculas,docentes,turmas,alunos_por_docente,pct_matriculas_escolas_turmas_zero,escolas_docentes_nulos
60,1200328,1433.0,19.0,20.0,75.42,68.11,0
53,1200054,1405.0,24.0,29.0,58.54,59.36,0
68,1200435,1167.0,20.0,20.0,58.35,58.87,0
170,1501451,1959.0,37.0,58.0,52.95,28.94,0
59,1200302,4981.0,97.0,113.0,51.35,47.52,0



Ano: 2022 | Indicador: alunos_por_turma


,CO_MUNICIPIO,matriculas,docentes,turmas,alunos_por_turma,pct_matriculas_escolas_turmas_zero,escolas_docentes_nulos
60,1200328,1433.0,19.0,20.0,71.65,68.11,0
68,1200435,1167.0,20.0,20.0,58.35,58.87,0
2783,3146206,435.0,9.0,8.0,54.38,0.00,0
472,2101350,429.0,11.0,8.0,53.62,56.41,0
636,2111409,2218.0,50.0,43.0,51.58,57.44,0



Ano: 2023 | Indicador: alunos_por_docente


...

,CO_MUNICIPIO,matriculas,docentes,turmas,alunos_por_turma,pct_matriculas_escolas_turmas_zero,escolas_docentes_nulos
11200,1200328,1490.0,19.0,21.0,70.95,68.59,0
11208,1200435,994.0,17.0,16.0,62.12,55.23,0
11193,1200054,1340.0,25.0,27.0,49.63,55.52,0
11612,2101350,385.0,9.0,8.0,48.12,59.74,0
11199,1200302,4934.0,87.0,104.0,47.44,48.12,0



Ano: 2025 | Indicador: alunos_por_docente


,CO_MUNICIPIO,matriculas,docentes,turmas,alunos_por_docente,pct_matriculas_escolas_turmas_zero,escolas_docentes_nulos
16770,1200328,1479.0,22.0,22.0,67.23,68.42,0
16778,1200435,1009.0,18.0,19.0,56.06,53.32,0
16769,1200302,4692.0,90.0,94.0,52.13,48.40,0
16763,1200054,1360.0,29.0,29.0,46.90,52.13,0
17182,2101350,375.0,8.0,8.0,46.88,57.60,0



Ano: 2025 | Indicador: alunos_por_turma


,CO_MUNICIPIO,matriculas,docentes,turmas,alunos_por_turma,pct_matriculas_escolas_turmas_zero,escolas_docentes_nulos
16770,1200328,1479.0,22.0,22.0,67.23,68.42,0
16778,1200435,1009.0,18.0,19.0,53.11,53.32,0
16769,1200302,4692.0,90.0,94.0,49.91,48.40,0
16763,1200054,1360.0,29.0,29.0,46.90,52.13,0
17182,2101350,375.0,8.0,8.0,46.88,57.60,0


A investigação identificou municípios que apresentam valores extremos recorrentes entre 2022 e 2025. Em parte desses municípios, observa-se elevada participação de matrículas vinculadas a escolas com turmas zeradas, o que pode influenciar as razões municipais de alunos por docente e alunos por turma. Entretanto, também foram identificados valores extremos sem essa característica. Dessa forma, os registros serão preservados, e os indicadores de cobertura poderão auxiliar na interpretação dos resultados durante a modelagem.

### 6.5 Associação entre cobertura e indicadores municipais

A investigação dos valores extremos identificou municípios
com razões elevadas de alunos por docente e alunos por turma,
frequentemente acompanhadas por alta participação de matrículas
em escolas com turmas zeradas.

Nesta etapa, será avaliada a associação entre essas variáveis,
utilizando a correlação de Spearman.

Essa medida permite investigar relações monotônicas sem
pressupor uma relação linear entre os indicadores.

A correlação não será interpretada como evidência de causalidade.

In [58]:
# Associação entre cobertura e indicadores municipais

resultados_correlacao = []

for ano, df_ano in df_indicadores_municipais.groupby("Ano"):

    cobertura = df_ano[
        "pct_matriculas_escolas_turmas_zero"
    ]

    for indicador in [
        "alunos_por_docente",
        "alunos_por_turma"
    ]:

        correlacao = cobertura.corr(
            df_ano[indicador],
            method="spearman"
        )

        resultados_correlacao.append({
            "Ano": ano,
            "Indicador": indicador,
            "Correlação Spearman": round(correlacao, 3)
        })

df_correlacao = pd.DataFrame(resultados_correlacao)

display(df_correlacao)

,Ano,Indicador,Correlação Spearman
0,2022,alunos_por_docente,0.323
1,2022,alunos_por_turma,0.342
2,2023,alunos_por_docente,0.328
3,2023,alunos_por_turma,0.343
4,2024,alunos_por_docente,0.308
5,2024,alunos_por_turma,0.328
6,2025,alunos_por_docente,0.315
7,2025,alunos_por_turma,0.326


### 6.6 Conclusão da análise dos indicadores municipais

Os indicadores de alunos por docente e alunos por turma
apresentaram distribuições relativamente estáveis entre
2022 e 2024, com alguns valores extremos.

A investigação identificou uma associação positiva entre
essas razões e a participação de matrículas em escolas
com turmas zeradas.

As correlações de Spearman variaram entre 0,308 e 0,343,
indicando uma associação positiva moderada e persistente
nos três anos.

Essa associação não demonstra causalidade e pode refletir,
em parte, relações matemáticas entre as variáveis utilizadas
na construção dos indicadores.

**Decisão metodológica:**

- Preservar os totais municipais de matrículas, docentes e turmas.
- Manter alunos por docente e alunos por turma como
  indicadores candidatos, com ressalvas.
- Criar pct_matriculas_escolas_turmas_zero como indicador
  de atenção à cobertura.
- Não excluir municípios ou corrigir zeros automaticamente.
- Avaliar posteriormente a contribuição dessas características
  para a modelagem da alfabetização.

## 7. Consolidação das características municipais

A partir das validações realizadas na análise exploratória,
será construída uma base municipal para cada ano do Censo
Escolar (2022, 2023 e 2024).

Serão consideradas escolas em funcionamento com pelo menos
uma matrícula nos anos iniciais do Ensino Fundamental.

As características contemplam infraestrutura escolar,
localização, matrículas, docentes, turmas e indicadores
de atenção à cobertura.

Os dados serão agregados por município e ano, preservando
os valores originais sem exclusões adicionais.

In [59]:
# Colunas necessárias para construir a base municipal

colunas_gold = [
    "CO_MUNICIPIO",
    "TP_SITUACAO_FUNCIONAMENTO",
    "TP_LOCALIZACAO",
    "QT_MAT_FUND_AI",
    "QT_MAT_FUND_AI_INT",
    "QT_DOC_FUND_AI",
    "QT_TUR_FUND_AI",
    "IN_INTERNET",
    "IN_BIBLIOTECA",
    "IN_SALA_LEITURA"
]

bases_municipais = []

for ano in [2022, 2023, 2024, 2025]:

    if ano < 2025:
        # Até 2024: arquivo único por ano
        arquivo = DADOS_DIR / f"{ano}_microdados_ed_basica.csv"

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=colunas_gold,
            low_memory=False
        )

    else:
        # 2025: informações distribuídas em quatro tabelas

        df_escolas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Escola_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "CO_MUNICIPIO",
                "TP_SITUACAO_FUNCIONAMENTO",
                "TP_LOCALIZACAO",
                "IN_INTERNET",
                "IN_BIBLIOTECA",
                "IN_SALA_LEITURA"
            ],
            low_memory=False
        )

        df_matriculas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Matricula_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_MAT_FUND_AI",
                "QT_MAT_FUND_AI_INT"
            ],
            low_memory=False
        )

        df_docentes = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Docente_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_DOC_FUND_AI"
            ],
            low_memory=False
        )

        df_turmas = pd.read_csv(
            DADOS_DIR / "2025_Tabela_Turma_V2.csv",
            sep=";",
            encoding="latin1",
            usecols=[
                "CO_ENTIDADE",
                "QT_TUR_FUND_AI"
            ],
            low_memory=False
        )

        # Integração por escola
        df = (
            df_escolas
            .merge(
                df_matriculas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_docentes,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
            .merge(
                df_turmas,
                on="CO_ENTIDADE",
                how="left",
                validate="one_to_one"
            )
        )

        del df_escolas, df_matriculas, df_docentes, df_turmas

    # Seleciona escolas ativas com matrículas nos anos iniciais
    df = df.loc[
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["QT_MAT_FUND_AI"] > 0)
    ].copy()

    # Indicadores auxiliares por escola
    df["escola_rural"] = (
        df["TP_LOCALIZACAO"] == 2
    ).astype(int)

    df["espaco_leitura"] = (
        (df["IN_BIBLIOTECA"] == 1) |
        (df["IN_SALA_LEITURA"] == 1)
    ).astype(int)

    df["matriculas_turmas_zero"] = (
        df["QT_MAT_FUND_AI"].where(
            df["QT_TUR_FUND_AI"] == 0, 0
        )
    )

    df["docentes_nulos"] = (
        df["QT_DOC_FUND_AI"].isna().astype(int)
    )

    # Consolidação municipal
    municipal = (
        df.groupby("CO_MUNICIPIO")
        .agg(
            qtd_escolas=("QT_MAT_FUND_AI", "size"),
            total_matriculas=("QT_MAT_FUND_AI", "sum"),
            total_docentes=("QT_DOC_FUND_AI", "sum"),
            total_turmas=("QT_TUR_FUND_AI", "sum"),
            escolas_rurais=("escola_rural", "sum"),
            escolas_internet=("IN_INTERNET", "sum"),
            escolas_biblioteca=("IN_BIBLIOTECA", "sum"),
            escolas_sala_leitura=("IN_SALA_LEITURA", "sum"),
            escolas_espaco_leitura=("espaco_leitura", "sum"),
            matriculas_integral=("QT_MAT_FUND_AI_INT", "sum"),
            matriculas_turmas_zero=("matriculas_turmas_zero", "sum"),
            qtd_escolas_docentes_nulos=("docentes_nulos", "sum")
        )
        .reset_index()
    )

    municipal["ano"] = ano

    municipal = municipal.rename(
        columns={"CO_MUNICIPIO": "id_municipio"}
    )

    # Percentuais de infraestrutura e localização
    for destino, origem in {
        "pct_escolas_rurais": "escolas_rurais",
        "pct_escolas_internet": "escolas_internet",
        "pct_escolas_biblioteca": "escolas_biblioteca",
        "pct_escolas_sala_leitura": "escolas_sala_leitura",
        "pct_escolas_espaco_leitura": "escolas_espaco_leitura"
    }.items():
        municipal[destino] = (
            municipal[origem] /
            municipal["qtd_escolas"] * 100
        )

    # Percentual de matrículas em tempo integral
    municipal["pct_matriculas_integral"] = (
        municipal["matriculas_integral"] /
        municipal["total_matriculas"] * 100
    )

    # Razões municipais
    municipal["alunos_por_docente"] = (
        municipal["total_matriculas"] /
        municipal["total_docentes"].replace(0, float("nan"))
    )

    municipal["alunos_por_turma"] = (
        municipal["total_matriculas"] /
        municipal["total_turmas"].replace(0, float("nan"))
    )

    # Indicador de cobertura dos registros de turmas
    municipal["pct_matriculas_escolas_turmas_zero"] = (
        municipal["matriculas_turmas_zero"] /
        municipal["total_matriculas"] * 100
    )

    # Estrutura final da base municipal
    colunas_saida = [
        "ano",
        "id_municipio",
        "qtd_escolas",
        "total_matriculas",
        "total_docentes",
        "total_turmas",
        "pct_escolas_rurais",
        "pct_escolas_internet",
        "pct_escolas_biblioteca",
        "pct_escolas_sala_leitura",
        "pct_escolas_espaco_leitura",
        "pct_matriculas_integral",
        "alunos_por_docente",
        "alunos_por_turma",
        "pct_matriculas_escolas_turmas_zero",
        "qtd_escolas_docentes_nulos"
    ]

    municipal = municipal[colunas_saida]

    bases_municipais.append(municipal)

    print(f"{ano}: {len(municipal):,} municípios")

    del df, municipal

2022: 5,570 municípios
2023: 5,570 municípios
2024: 5,570 municípios
2025: 5,571 municípios


### 7.1 Validação das bases municipais consolidadas

Antes da exportação, serão verificadas:

- Quantidade de municípios por ano.
- Unicidade da chave município/ano.
- Presença de valores nulos.
- Percentuais fora do intervalo de 0 a 100.
- Quantidades negativas.

A exportação será realizada somente após a validação
dos resultados.

In [60]:
# Validação final das bases municipais

colunas_percentuais = [
    "pct_escolas_rurais",
    "pct_escolas_internet",
    "pct_escolas_biblioteca",
    "pct_escolas_sala_leitura",
    "pct_escolas_espaco_leitura",
    "pct_matriculas_integral",
    "pct_matriculas_escolas_turmas_zero"
]

colunas_quantitativas = [
    "qtd_escolas",
    "total_matriculas",
    "total_docentes",
    "total_turmas",
    "qtd_escolas_docentes_nulos"
]

resultados_validacao = []

for base in bases_municipais:
    ano = int(base["ano"].iloc[0])

    duplicados = base.duplicated(
        subset=["ano", "id_municipio"]
    ).sum()

    nulos = base.isna().sum().sum()

    percentuais_invalidos = (
        (~base[colunas_percentuais].between(0, 100))
        if False else
        (
            (base[colunas_percentuais] < 0) |
            (base[colunas_percentuais] > 100)
        )
    ).sum().sum()

    negativos = (
        base[colunas_quantitativas] < 0
    ).sum().sum()

    resultados_validacao.append({
        "Ano": ano,
        "Municípios": len(base),
        "Duplicados": int(duplicados),
        "Valores nulos": int(nulos),
        "Percentuais inválidos": int(percentuais_invalidos),
        "Quantidades negativas": int(negativos)
    })

df_validacao_final = pd.DataFrame(resultados_validacao)

display(df_validacao_final)

,Ano,Municípios,Duplicados,Valores nulos,Percentuais inválidos,Quantidades negativas
0,2022,5570,0,0,0,0
1,2023,5570,0,0,0,0
2,2024,5570,0,0,0,0
3,2025,5571,0,0,0,0


### 7.2 Exportação das bases municipais

Após as validações de integridade, as bases municipais
serão exportadas em arquivos CSV separados por ano.

Cada arquivo contém 5.570 municípios e 16 colunas,
com granularidade município/ano.

Os arquivos serão utilizados como insumo para a
construção da camada Gold do Tech Challenge.

A exportação utiliza UTF-8, separador de vírgula
e ponto como separador decimal.

In [61]:
# Exportação das bases municipais

SAIDA_DIR = BASE_DIR / "saida"
SAIDA_DIR.mkdir(parents=True, exist_ok=True)

for base in bases_municipais:
    ano = int(base["ano"].iloc[0])

    arquivo_saida = (
        SAIDA_DIR / f"censo_escolar_municipal_{ano}.csv"
    )

    base.to_csv(
        arquivo_saida,
        index=False,
        sep=",",
        encoding="utf-8",
        float_format="%.6f"
    )

    # Conferência do arquivo gerado
    conferencia = pd.read_csv(
        arquivo_saida,
        dtype={"id_municipio": "string"}
    )

    assert conferencia.shape == base.shape, (
        f"Dimensões divergentes em {ano}"
    )

    assert not conferencia.duplicated(
        subset=["ano", "id_municipio"]
    ).any(), f"Chaves duplicadas em {ano}"

    print(
        f"{arquivo_saida.name} | "
        f"{conferencia.shape[0]} registros | "
        f"{conferencia.shape[1]} colunas"
    )

censo_escolar_municipal_2022.csv | 5570 registros | 16 colunas
censo_escolar_municipal_2023.csv | 5570 registros | 16 colunas
censo_escolar_municipal_2024.csv | 5570 registros | 16 colunas
censo_escolar_municipal_2025.csv | 5571 registros | 16 colunas


## 8. Adequação do Censo Escolar 2025

A partir de 2025, os microdados foram disponibilizados
em tabelas separadas por entidade.

Antes da integração, será necessário identificar a
granularidade de cada tabela e suas chaves de relacionamento.

O objetivo é reconstruir os indicadores municipais
selecionados na EDA de 2022 a 2024, respeitando as
particularidades da estrutura de 2025.

In [62]:
# Inspeção inicial das tabelas de 2025

arquivos_2025 = {
    "Escola": "2025_Tabela_Escola_V2.csv",
    "Matrícula": "2025_Tabela_Matricula_V2.csv",
    "Turma": "2025_Tabela_Turma_V2.csv",
    "Docente": "2025_Tabela_Docente_V2.csv"
}

for nome, arquivo in arquivos_2025.items():

    df = pd.read_csv(
        DADOS_DIR / arquivo,
        sep=";",
        encoding="latin1",
        nrows=1000,
        low_memory=False
    )

    print(f"\n{'=' * 50}")
    print(f"TABELA: {nome}")
    print(f"{'=' * 50}")

    print(f"Colunas: {len(df.columns)}")

    print("\nChaves encontradas:")
    print([
        coluna for coluna in df.columns
        if coluna.startswith("CO_")
        or coluna.startswith("ID_")
    ])

    if "CO_ENTIDADE" in df.columns:
        print(
            "\nCO_ENTIDADE únicos na amostra:",
            df["CO_ENTIDADE"].nunique()
        )

        print(
            "Registros na amostra:",
            len(df)
        )

    del df


TABELA: Escola
Colunas: 290

Chaves encontradas:
['CO_REGIAO', 'CO_UF', 'CO_MUNICIPIO', 'CO_REGIAO_GEOG_INTERM', 'CO_REGIAO_GEOG_IMED', 'CO_DISTRITO', 'CO_SUBDISTRITO', 'CO_ENTIDADE', 'CO_REDE', 'CO_ORGAO_REGIONAL', 'CO_ESCOLA_SEDE_VINCULADA', 'CO_IES_OFERTANTE', 'CO_LINGUA_INDIGENA_1', 'CO_LINGUA_INDIGENA_2', 'CO_LINGUA_INDIGENA_3']

CO_ENTIDADE únicos na amostra: 1000
Registros na amostra: 1000

TABELA: Matrícula
Colunas: 263

Chaves encontradas:
['CO_REGIAO', 'CO_UF', 'CO_MUNICIPIO', 'CO_REGIAO_GEOG_INTERM', 'CO_REGIAO_GEOG_IMED', 'CO_DISTRITO', 'CO_SUBDISTRITO', 'CO_ENTIDADE', 'CO_REDE']

CO_ENTIDADE únicos na amostra: 1000
Registros na amostra: 1000

TABELA: Turma
Colunas: 218

Chaves encontradas:
['CO_REGIAO', 'CO_UF', 'CO_MUNICIPIO', 'CO_REGIAO_GEOG_INTERM', 'CO_REGIAO_GEOG_IMED', 'CO_DISTRITO', 'CO_SUBDISTRITO', 'CO_ENTIDADE', 'CO_REDE']

CO_ENTIDADE únicos na amostra: 1000
Registros na amostra: 1000

TABELA: Docente
Colunas: 184

Chaves encontradas:
['CO_REGIAO', 'CO_UF', 'CO

### 8.1 Validação da granularidade das tabelas de 2025

A análise inicial identificou CO_ENTIDADE nas quatro tabelas.

Entretanto, a unicidade observada na amostra não garante
que cada escola possua apenas um registro.

Nesta etapa, verificamos a quantidade total de registros,
escolas distintas e registros adicionais por escola.

O objetivo é definir a estratégia de integração sem
multiplicar indevidamente as informações.

In [63]:
# Validação das chaves nas tabelas completas

resultados_chaves = []

for nome, arquivo in arquivos_2025.items():

    df = pd.read_csv(
        DADOS_DIR / arquivo,
        sep=";",
        encoding="latin1",
        usecols=["CO_ENTIDADE"],
        low_memory=False
    )

    total_registros = len(df)
    escolas_distintas = df["CO_ENTIDADE"].nunique()

    resultados_chaves.append({
        "Tabela": nome,
        "Registros": total_registros,
        "Escolas distintas": escolas_distintas,
        "Registros adicionais": total_registros - escolas_distintas,
        "Máximo por escola": (
            df["CO_ENTIDADE"].value_counts().max()
        )
    })

    del df

df_chaves_2025 = pd.DataFrame(resultados_chaves)

display(df_chaves_2025)

,Tabela,Registros,Escolas distintas,Registros adicionais,Máximo por escola
0,Escola,214192,214192,0,1
1,Matrícula,178766,178766,0,1
2,Turma,178772,178772,0,1
3,Docente,178772,178772,0,1


### 8.2 Identificação das variáveis de 2025

Com a unicidade de CO_ENTIDADE confirmada, será realizado
o mapeamento das variáveis necessárias para reproduzir
os indicadores municipais de 2022 a 2024.

O objetivo é identificar os campos equivalentes nas
tabelas Escola, Matrícula, Turma e Docente.

A equivalência semântica será posteriormente conferida
no dicionário de dados de 2025.

In [64]:
# Identificação das colunas relevantes de 2025

termos = [
    "SITUACAO_FUNCIONAMENTO",
    "LOCALIZACAO",
    "IN_INTERNET",
    "IN_BIBLIOTECA",
    "IN_SALA_LEITURA",
    "FUND_AI",
    "FUNDAMENTAL",
    "INTEGRAL"
]

for nome, arquivo in arquivos_2025.items():

    colunas = pd.read_csv(
        DADOS_DIR / arquivo,
        sep=";",
        encoding="latin1",
        nrows=0
    ).columns.tolist()

    encontradas = [
        coluna for coluna in colunas
        if any(termo in coluna for termo in termos)
    ]

    print(f"\n{'=' * 50}")
    print(f"TABELA: {nome}")
    print(f"{'=' * 50}")

    for coluna in encontradas:
        print(coluna)


TABELA: Escola
TP_LOCALIZACAO
TP_LOCALIZACAO_DIFERENCIADA
TP_SITUACAO_FUNCIONAMENTO
IN_BIBLIOTECA
IN_BIBLIOTECA_SALA_LEITURA
IN_SALA_LEITURA
IN_INTERNET
IN_INTERNET_ALUNOS
IN_INTERNET_ADMINISTRATIVO
IN_INTERNET_APRENDIZAGEM
IN_INTERNET_COMUNIDADE
IN_COMUM_FUND_AI
IN_ESP_EXCLUSIVA_FUND_AI

TABELA: Matrícula
TP_LOCALIZACAO
TP_LOCALIZACAO_DIFERENCIADA
QT_MAT_FUND_AI
QT_MAT_FUND_AI_1
QT_MAT_FUND_AI_2
QT_MAT_FUND_AI_3
QT_MAT_FUND_AI_4
QT_MAT_FUND_AI_5
QT_MAT_EJA_FUND_AI
QT_MAT_ESP_FUND_AI
QT_MAT_ESP_CC_FUND_AI
QT_MAT_ESP_CE_FUND_AI
QT_MAT_FUND_AI_D
QT_MAT_FUND_AI_DM
QT_MAT_FUND_AI_DV
QT_MAT_FUND_AI_N
QT_MAT_FUND_AI_INT

TABELA: Turma
TP_LOCALIZACAO
TP_LOCALIZACAO_DIFERENCIADA
QT_TUR_FUND_AI
QT_TUR_FUND_AI_1
QT_TUR_FUND_AI_2
QT_TUR_FUND_AI_3
QT_TUR_FUND_AI_4
QT_TUR_FUND_AI_5
QT_TUR_FUND_AI_MULTIETAPA
QT_TUR_EJA_FUND_AI
QT_TUR_FUND_AI_D
QT_TUR_FUND_AI_DM
QT_TUR_FUND_AI_DV
QT_TUR_FUND_AI_N
QT_TUR_FUND_AI_INT

TABELA: Docente
TP_LOCALIZACAO
TP_LOCALIZACAO_DIFERENCIADA
QT_DOC_FUND_AI
QT_DOC_FUN

### 8.3 Integração das tabelas de 2025

As tabelas Escola, Matrícula, Turma e Docente apresentam
CO_ENTIDADE como chave única.

A integração será realizada utilizando a tabela Escola
como referência, com relacionamentos do tipo left join.

Serão selecionadas apenas as colunas necessárias para
reproduzir os indicadores municipais de 2022 a 2024.

Nesta etapa, nenhuma escola será excluída e nenhum
valor ausente será substituído automaticamente.

### 8.3 Integração das tabelas de 2025

As tabelas Escola, Matrícula, Turma e Docente apresentam
CO_ENTIDADE como chave única.

A integração será realizada utilizando a tabela Escola
como referência, com relacionamentos do tipo left join.

Serão selecionadas apenas as colunas necessárias para
reproduzir os indicadores municipais de 2022 a 2024.

Nesta etapa, nenhuma escola será excluída e nenhum
valor ausente será substituído automaticamente.

In [65]:
# Integração das tabelas do Censo Escolar 2025

colunas_2025 = {
    "Escola": [
        "CO_ENTIDADE",
        "CO_MUNICIPIO",
        "TP_SITUACAO_FUNCIONAMENTO",
        "TP_LOCALIZACAO",
        "IN_INTERNET",
        "IN_BIBLIOTECA",
        "IN_SALA_LEITURA"
    ],
    "Matrícula": [
        "CO_ENTIDADE",
        "QT_MAT_FUND_AI",
        "QT_MAT_FUND_AI_INT"
    ],
    "Turma": [
        "CO_ENTIDADE",
        "QT_TUR_FUND_AI"
    ],
    "Docente": [
        "CO_ENTIDADE",
        "QT_DOC_FUND_AI"
    ]
}

tabelas_2025 = {}

for nome, arquivo in arquivos_2025.items():

    tabelas_2025[nome] = pd.read_csv(
        DADOS_DIR / arquivo,
        sep=";",
        encoding="latin1",
        usecols=colunas_2025[nome],
        low_memory=False
    )

df_2025 = tabelas_2025["Escola"].copy()

for nome in ["Matrícula", "Turma", "Docente"]:

    df_2025 = df_2025.merge(
        tabelas_2025[nome],
        on="CO_ENTIDADE",
        how="left",
        validate="one_to_one"
    )

print("Registros:", len(df_2025))
print("Escolas distintas:", df_2025["CO_ENTIDADE"].nunique())
print("Colunas:", len(df_2025.columns))

display(df_2025.head())

Registros: 214192
Escolas distintas: 214192
Colunas: 11


,CO_MUNICIPIO,CO_ENTIDADE,TP_LOCALIZACAO,TP_SITUACAO_FUNCIONAMENTO,IN_BIBLIOTECA,IN_SALA_LEITURA,IN_INTERNET,QT_MAT_FUND_AI,QT_MAT_FUND_AI_INT,QT_TUR_FUND_AI,QT_DOC_FUND_AI
0,1100015,11022558,2,1,0.0,0.0,1.0,4.0,0.0,4.0,1.0
1,1100015,11024275,1,1,1.0,1.0,1.0,0.0,0.0,0.0,0.0
2,1100015,11024291,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1100015,11024666,2,1,0.0,1.0,1.0,80.0,0.0,5.0,6.0
4,1100015,11024682,1,1,1.0,1.0,1.0,0.0,0.0,0.0,0.0


### 8.4 Seleção e validação das escolas de 2025

Serão aplicados os mesmos critérios utilizados
na análise dos anos anteriores:

- Escolas em funcionamento.
- Pelo menos uma matrícula nos anos iniciais
  do Ensino Fundamental.

Após a seleção, serão verificadas a cobertura
municipal e a presença de valores ausentes nas
variáveis necessárias à construção dos indicadores.

Nenhum valor ausente será substituído automaticamente.

In [66]:
# Seleção das escolas de 2025

df_2025_selecionado = df_2025.loc[
    (df_2025["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
    (df_2025["QT_MAT_FUND_AI"] > 0)
].copy()

print("Escolas selecionadas:", len(df_2025_selecionado))

print(
    "Municípios distintos:",
    df_2025_selecionado["CO_MUNICIPIO"].nunique()
)

# Verificação de valores ausentes

colunas_validacao = [
    "TP_LOCALIZACAO",
    "IN_INTERNET",
    "IN_BIBLIOTECA",
    "IN_SALA_LEITURA",
    "QT_MAT_FUND_AI",
    "QT_MAT_FUND_AI_INT",
    "QT_DOC_FUND_AI",
    "QT_TUR_FUND_AI"
]

display(
    df_2025_selecionado[
        colunas_validacao
    ].isna().sum().to_frame("Valores nulos")
)

Escolas selecionadas: 101292
Municípios distintos: 5571


,Valores nulos
TP_LOCALIZACAO,0
IN_INTERNET,0
IN_BIBLIOTECA,0
IN_SALA_LEITURA,0
QT_MAT_FUND_AI,0
QT_MAT_FUND_AI_INT,0
QT_DOC_FUND_AI,0
QT_TUR_FUND_AI,0


### 8.5 Validação da cobertura municipal

A base de 2025 apresentou 5.571 municípios, enquanto
as bases de 2022 a 2024 apresentaram 5.570.

Será realizada uma comparação dos códigos municipais
entre 2024 e 2025 para identificar possíveis alterações
na cobertura territorial.

Nenhum município será excluído sem investigação.

In [67]:
# Comparação dos municípios de 2024 e 2025

df_2024 = pd.read_csv(
    SAIDA_DIR / "censo_escolar_municipal_2024.csv"
)

municipios_2024 = set(df_2024["id_municipio"].astype(int))
municipios_2025 = set(df_2025_selecionado["CO_MUNICIPIO"].astype(int))

print("Municípios presentes apenas em 2025:")
print(sorted(municipios_2025 - municipios_2024))

print("\nMunicípios de 2024 ausentes em 2025:")
print(sorted(municipios_2024 - municipios_2025))

Municípios presentes apenas em 2025:
[5101837]

Municípios de 2024 ausentes em 2025:
[]


### 8.6 Validação dos quantitativos de 2025

Após a seleção das escolas, serão avaliadas as
inconsistências e particularidades dos quantitativos
de matrículas, docentes e turmas.

As validações seguem os mesmos critérios utilizados
na análise de 2022 a 2024.

Valores zerados serão preservados e documentados,
sem exclusão ou substituição automática.

In [68]:
# Validação dos quantitativos de 2025

validacao_2025 = {
    "Escolas": len(df_2025_selecionado),

    "Valores negativos": (
        df_2025_selecionado[
            [
                "QT_MAT_FUND_AI",
                "QT_MAT_FUND_AI_INT",
                "QT_DOC_FUND_AI",
                "QT_TUR_FUND_AI"
            ]
        ].lt(0).sum().sum()
    ),

    "Matrículas integrais > total": (
        df_2025_selecionado["QT_MAT_FUND_AI_INT"]
        > df_2025_selecionado["QT_MAT_FUND_AI"]
    ).sum(),

    "Docentes zerados": (
        df_2025_selecionado["QT_DOC_FUND_AI"] == 0
    ).sum(),

    "Turmas zeradas": (
        df_2025_selecionado["QT_TUR_FUND_AI"] == 0
    ).sum(),

    "Docentes e turmas zerados": (
        (df_2025_selecionado["QT_DOC_FUND_AI"] == 0) &
        (df_2025_selecionado["QT_TUR_FUND_AI"] == 0)
    ).sum()
}

display(pd.DataFrame([validacao_2025]))

,Escolas,Valores negativos,Matrículas integrais > total,Docentes zerados,Turmas zeradas,Docentes e turmas zerados
0,101292,0,0,13399,13399,13399


### 8.7 Distribuição das escolas sem turmas por localização

Será analisada a distribuição das escolas com matrículas
nos anos iniciais, mas sem turmas registradas.

A análise permitirá comparar o comportamento de 2025
com os resultados observados entre 2022 e 2024,
especialmente nas escolas rurais.

In [69]:
# Distribuição das escolas sem turmas por localização

df_validacao = df_2025_selecionado.copy()

df_validacao["turmas_zeradas"] = (
    df_validacao["QT_TUR_FUND_AI"] == 0
)

resultado = (
    df_validacao
    .groupby("TP_LOCALIZACAO")
    .agg(
        escolas=("CO_ENTIDADE", "count"),
        turmas_zeradas=("turmas_zeradas", "sum")
    )
    .reset_index()
)

resultado["pct_turmas_zeradas"] = (
    resultado["turmas_zeradas"]
    / resultado["escolas"]
    * 100
).round(2)

resultado["localizacao"] = resultado[
    "TP_LOCALIZACAO"
].map({
    1: "Urbana",
    2: "Rural"
})

display(
    resultado[
        [
            "localizacao",
            "escolas",
            "turmas_zeradas",
            "pct_turmas_zeradas"
        ]
    ]
)

,localizacao,escolas,turmas_zeradas,pct_turmas_zeradas
0,Urbana,59771,814,1.36
1,Rural,41521,12585,30.31


### 8.8 Preparação das variáveis para agregação municipal

Com as validações exploratórias concluídas, serão
preparadas as variáveis auxiliares necessárias à
construção dos indicadores municipais de 2025.

Serão mantidas as mesmas regras utilizadas entre
2022 e 2024, garantindo a consistência metodológica
da série histórica.

A agregação será realizada posteriormente,
utilizando CO_MUNICIPIO como chave.

In [70]:
# Preparação das variáveis auxiliares de 2025

df_2025_preparado = df_2025_selecionado.copy()

# Identificação de escolas rurais
df_2025_preparado["escola_rural"] = (
    df_2025_preparado["TP_LOCALIZACAO"] == 2
).astype(int)

# Infraestrutura escolar
df_2025_preparado["escola_internet"] = (
    df_2025_preparado["IN_INTERNET"] == 1
).astype(int)

df_2025_preparado["escola_biblioteca"] = (
    df_2025_preparado["IN_BIBLIOTECA"] == 1
).astype(int)

df_2025_preparado["escola_sala_leitura"] = (
    df_2025_preparado["IN_SALA_LEITURA"] == 1
).astype(int)

df_2025_preparado["escola_espaco_leitura"] = (
    (df_2025_preparado["IN_BIBLIOTECA"] == 1) |
    (df_2025_preparado["IN_SALA_LEITURA"] == 1)
).astype(int)

# Matrículas em escolas sem turmas
df_2025_preparado["matriculas_turmas_zero"] = (
    df_2025_preparado["QT_MAT_FUND_AI"].where(
        df_2025_preparado["QT_TUR_FUND_AI"] == 0,
        0
    )
)

# Identificação de docentes ausentes
df_2025_preparado["docentes_nulos"] = (
    df_2025_preparado["QT_DOC_FUND_AI"].isna()
).astype(int)

print("Escolas preparadas:", len(df_2025_preparado))

print(
    "Municípios:",
    df_2025_preparado["CO_MUNICIPIO"].nunique()
)

print(
    "Variáveis auxiliares criadas:",
    [
        "escola_rural",
        "escola_internet",
        "escola_biblioteca",
        "escola_sala_leitura",
        "escola_espaco_leitura",
        "matriculas_turmas_zero",
        "docentes_nulos"
    ]
)

Escolas preparadas:

 101292
Municípios: 5571
Variáveis auxiliares criadas: ['escola_rural', 'escola_internet', 'escola_biblioteca', 'escola_sala_leitura', 'escola_espaco_leitura', 'matriculas_turmas_zero', 'docentes_nulos']


### 8.9 Agregação municipal de 2025

Os registros das escolas selecionadas serão agregados
por município, utilizando CO_MUNICIPIO como chave.

Serão calculados os mesmos indicadores produzidos
para 2022, 2023 e 2024, preservando a metodologia
e a estrutura da base municipal.

Os indicadores representam características agregadas
das escolas com matrículas nos anos iniciais do
Ensino Fundamental.

In [71]:
# Agregação municipal de 2025

df_municipal_2025 = (
    df_2025_preparado
    .groupby("CO_MUNICIPIO")
    .agg(
        qtd_escolas=("CO_ENTIDADE", "count"),
        total_matriculas=("QT_MAT_FUND_AI", "sum"),
        total_docentes=("QT_DOC_FUND_AI", "sum"),
        total_turmas=("QT_TUR_FUND_AI", "sum"),

        escolas_rurais=("escola_rural", "sum"),
        escolas_internet=("escola_internet", "sum"),
        escolas_biblioteca=("escola_biblioteca", "sum"),
        escolas_sala_leitura=("escola_sala_leitura", "sum"),
        escolas_espaco_leitura=("escola_espaco_leitura", "sum"),

        matriculas_integral=("QT_MAT_FUND_AI_INT", "sum"),
        matriculas_turmas_zero=("matriculas_turmas_zero", "sum"),
        qtd_escolas_docentes_nulos=("docentes_nulos", "sum")
    )
    .reset_index()
    .rename(columns={"CO_MUNICIPIO": "id_municipio"})
)

# Percentuais de infraestrutura e localização

indicadores_percentuais = {
    "pct_escolas_rurais": "escolas_rurais",
    "pct_escolas_internet": "escolas_internet",
    "pct_escolas_biblioteca": "escolas_biblioteca",
    "pct_escolas_sala_leitura": "escolas_sala_leitura",
    "pct_escolas_espaco_leitura": "escolas_espaco_leitura"
}

for indicador, coluna in indicadores_percentuais.items():
    df_municipal_2025[indicador] = (
        df_municipal_2025[coluna]
        / df_municipal_2025["qtd_escolas"]
        * 100
    )

# Percentual de matrículas em tempo integral

df_municipal_2025["pct_matriculas_integral"] = (
    df_municipal_2025["matriculas_integral"]
    / df_municipal_2025["total_matriculas"]
    * 100
)

# Razões de alunos por docente e turma

df_municipal_2025["alunos_por_docente"] = (
    df_municipal_2025["total_matriculas"]
    / df_municipal_2025["total_docentes"].replace(0, float("nan"))
)

df_municipal_2025["alunos_por_turma"] = (
    df_municipal_2025["total_matriculas"]
    / df_municipal_2025["total_turmas"].replace(0, float("nan"))
)

# Percentual de matrículas em escolas sem turmas

df_municipal_2025["pct_matriculas_escolas_turmas_zero"] = (
    df_municipal_2025["matriculas_turmas_zero"]
    / df_municipal_2025["total_matriculas"]
    * 100
)

# Ano de referência

df_municipal_2025["ano"] = 2025

# Seleção e ordenação dos 16 campos

colunas_finais = [
    "ano",
    "id_municipio",
    "qtd_escolas",
    "total_matriculas",
    "total_docentes",
    "total_turmas",
    "pct_escolas_rurais",
    "pct_escolas_internet",
    "pct_escolas_biblioteca",
    "pct_escolas_sala_leitura",
    "pct_escolas_espaco_leitura",
    "pct_matriculas_integral",
    "alunos_por_docente",
    "alunos_por_turma",
    "pct_matriculas_escolas_turmas_zero",
    "qtd_escolas_docentes_nulos"
]

df_municipal_2025 = df_municipal_2025[colunas_finais]

print("Municípios:", len(df_municipal_2025))
print("Colunas:", len(df_municipal_2025.columns))

display(df_municipal_2025.head())

Municípios: 5571
Colunas: 16


,ano,id_municipio,qtd_escolas,total_matriculas,total_docentes,total_turmas,pct_escolas_rurais,pct_escolas_internet,pct_escolas_biblioteca,pct_escolas_sala_leitura,pct_escolas_espaco_leitura,pct_matriculas_integral,alunos_por_docente,alunos_por_turma,pct_matriculas_escolas_turmas_zero,qtd_escolas_docentes_nulos
0,2025,1100015,17,1694.0,86.0,90.0,82.352941,88.235294,5.882353,47.058824,52.941176,2.302243,19.697674,18.822222,5.489965,0
1,2025,1100023,28,7363.0,323.0,313.0,25.000000,100.000000,71.428571,21.428571,82.142857,10.484857,22.795666,23.523962,0.000000,0
2,2025,1100031,3,385.0,30.0,28.0,0.000000,100.000000,66.666667,0.000000,66.666667,0.000000,12.833333,13.750000,0.000000,0
3,2025,1100049,40,6223.0,316.0,310.0,47.500000,97.500000,50.000000,25.000000,55.000000,4.274466,19.693038,20.074194,0.000000,0
4,2025,1100056,6,1278.0,64.0,57.0,0.000000,100.000000,33.333333,33.333333,66.666667,7.511737,19.968750,22.421053,0.000000,0


### 8.10 Validação final da base municipal de 2025

A base municipal será submetida a verificações de
integridade, incluindo:

- Duplicidade de município e ano.
- Presença de valores nulos ou infinitos.
- Percentuais fora do intervalo de 0% a 100%.
- Quantidades negativas.
- Municípios sem docentes ou turmas registrados.

As verificações seguem os critérios adotados nas
bases municipais de 2022 a 2024.

In [72]:
import numpy as np

# Colunas percentuais
colunas_percentuais = [
    coluna for coluna in df_municipal_2025.columns
    if coluna.startswith("pct_")
]

# Colunas de quantidades
colunas_quantidades = [
    "qtd_escolas",
    "total_matriculas",
    "total_docentes",
    "total_turmas",
    "qtd_escolas_docentes_nulos"
]

validacao_final_2025 = {
    "Municípios": len(df_municipal_2025),

    "Duplicados": df_municipal_2025.duplicated(
        subset=["ano", "id_municipio"]
    ).sum(),

    "Valores nulos": df_municipal_2025.isna().sum().sum(),

    "Valores infinitos": np.isinf(
        df_municipal_2025.select_dtypes(include="number")
    ).sum().sum(),

    "Percentuais inválidos": (
        (df_municipal_2025[colunas_percentuais] < 0) |
        (df_municipal_2025[colunas_percentuais] > 100)
    ).sum().sum(),

    "Quantidades negativas": (
        df_municipal_2025[colunas_quantidades] < 0
    ).sum().sum(),

    "Municípios sem docentes": (
        df_municipal_2025["total_docentes"] == 0
    ).sum(),

    "Municípios sem turmas": (
        df_municipal_2025["total_turmas"] == 0
    ).sum()
}

display(pd.DataFrame([validacao_final_2025]))

,Municípios,Duplicados,Valores nulos,Valores infinitos,Percentuais inválidos,Quantidades negativas,Municípios sem docentes,Municípios sem turmas
0,5571,0,0,0,0,0,0,0


## 9. Conclusão da Análise Exploratória de Dados (EDA)

A Análise Exploratória de Dados do Censo Escolar, referente ao período de **2022 a 2025**, foi concluída com a padronização, consolidação e validação dos indicadores educacionais em nível municipal.

O objetivo desta etapa foi compreender a estrutura dos dados, identificar possíveis inconsistências e estabelecer critérios metodológicos para a construção posterior das camadas de dados do projeto.

### 9.1 Resultados consolidados

| Ano | Municípios | Indicadores | Status |
|---|---:|---:|---|
| 2022 | 5.570 | 16 | Validado |
| 2023 | 5.570 | 16 | Validado |
| 2024 | 5.570 | 16 | Validado |
| 2025 | 5.571 | 16 | Validado |
| **Total** | **22.281 registros** | **16 por registro** | **Concluído** |

Foram gerados quatro arquivos CSV, contendo os indicadores municipais de cada ano.

### 9.2 Principais atividades realizadas

- Identificação e comparação das variáveis disponíveis entre 2022 e 2025.
- Seleção de escolas em funcionamento com pelo menos uma matrícula nos anos iniciais do Ensino Fundamental.
- Adaptação do processamento à nova estrutura dos arquivos do Censo Escolar de 2025.
- Análise de valores ausentes, negativos e inconsistências nos quantitativos.
- Investigação de escolas com matrículas, mas sem docentes ou turmas registrados.
- Análise das diferenças entre escolas urbanas e rurais.
- Construção de 16 indicadores municipais padronizados.
- Validação de duplicidades, valores nulos, infinitos e percentuais inválidos.
- Exportação dos resultados municipais em arquivos CSV.

### 9.3 Principais observações

**Escolas sem docentes ou turmas registrados**

Foi identificada, em todos os anos analisados, a existência de escolas com matrículas nos anos iniciais do Ensino Fundamental, mas sem docentes ou turmas registrados.

Esse comportamento apresenta maior concentração nas escolas rurais e permaneceu relativamente estável ao longo do período.

Os registros foram preservados, considerando que sua exclusão poderia comprometer a representatividade dos dados.

Entretanto, essa característica exige cautela na interpretação dos indicadores `alunos_por_docente` e `alunos_por_turma`, especialmente em municípios com maior proporção de matrículas nessas escolas.

**Alteração na cobertura municipal de 2025**

A base de 2025 apresentou 5.571 municípios, enquanto os anos anteriores apresentaram 5.570.

A comparação identificou o código municipal `5101837` como adicional em 2025, sem ausência de municípios anteriormente representados.

O registro foi preservado. Sua identificação oficial e a natureza da alteração territorial ainda precisam ser confirmadas.

**Mudança estrutural dos arquivos de 2025**

Diferentemente dos anos anteriores, os dados de 2025 foram disponibilizados em tabelas separadas por domínio, como Escola, Matrícula, Turma e Docente.

Foi necessário integrar essas informações por meio da chave `CO_ENTIDADE`, preservando a granularidade de uma linha por escola.

Após a integração, foram aplicados os mesmos critérios de seleção e agregação utilizados nos anos anteriores.

### 9.4 Qualidade dos dados

As bases municipais dos quatro anos foram submetidas às seguintes verificações:

- Unicidade da combinação município e ano.
- Ausência de valores nulos e infinitos.
- Percentuais dentro do intervalo de 0% a 100%.
- Ausência de quantitativos negativos.
- Existência de matrículas, docentes e turmas nos totais municipais.

**Todas as bases municipais passaram nas validações estabelecidas.**

Isso não significa ausência de particularidades nos dados originais, mas indica que os resultados consolidados atendem aos critérios de qualidade definidos nesta análise.

### 9.5 Próximas etapas

Com a EDA concluída, o projeto seguirá para a construção do pipeline de engenharia de dados:

**Bronze → Silver → Gold**

- **Bronze:** armazenamento dos arquivos originais do INEP, preservando os dados de origem.
- **Silver:** integração, padronização, seleção e tratamento dos dados conforme as regras identificadas na EDA.
- **Gold:** construção dos indicadores municipais consolidados, preparados para integração com as demais fontes do Tech Challenge.

Os arquivos CSV produzidos durante a EDA serão utilizados como **referência para validar os resultados da camada Gold**, não substituindo os dados originais da Bronze.

### 9.6 Considerações finais

A análise permitiu estabelecer uma metodologia consistente para trabalhar com os dados do Censo Escolar entre 2022 e 2025, mesmo diante das diferenças estruturais entre os arquivos.

Os resultados fornecem uma base para a construção de indicadores educacionais municipais e sua posterior integração com outras fontes de dados.

A seleção definitiva das variáveis para os modelos de Machine Learning será realizada em uma etapa posterior, considerando sua qualidade, relevância e relação com o problema de previsão da alfabetização.

**Status da etapa: EDA concluída.**